# Exercise 4 — Using AI on Literature: Propose, Then Verify

**Biological Databases WS 2026/27 — Exercise 4**

Exercise 3 ended with an uncomfortable result: an AI assistant could not supply your identifiers,
because it does not have the release you downloaded and cannot show its work. The lesson was that
**an LLM cannot replace the database.**

This exercise asks the question that follows from it: *so where does it actually help?*

The answer, in research as it is actually done, is the literature loop:

> **The model proposes. The database disposes.**

You will run that loop yourself, on a small corpus of real abstracts, and you will find the failure
that matters most — not invented identifiers, but **real identifiers attached to papers that do not
say what the model claims**.

By the end you will have:

1. asked the same question of a local model **with and without** the papers in front of it;
2. verified every claim it made at **three increasing levels of strictness**;
3. found the cases where a citation is real, the quote is verbatim, and the claim is still wrong;
4. pushed every entity it extracted through **the linking cascade you built in Exercise 3**;
5. produced an evidence table you could act on, plus a **prompt log** that lets someone repeat it.

> **Nothing is submitted and nothing is collected.** Everything stays on your own machine, and you
> present your findings in class rather than handing anything in. The task boxes tell you what to
> explain and what to show.

## How the session works

The exercise is a single loop, run five times with increasing strictness. Each part fails in a way
the previous part could not detect — and that escalation *is* the content.

| Part | What you do | What it teaches |
|---|---|---|
| 1 | Call a local model, prove it is deterministic | a measurement, not a chat |
| 2 | Load your assigned ontology and the Ex3 linker | the referee |
| 3 | Build a corpus of abstracts for your topic | retrieval before reasoning |
| 4 | Ask the same question with and without context | grounding changes everything |
| 5 | Verify citations at three levels | verbatim is not support |
| 6 | Hunt hedged and negated mentions | `angiosarcoma-like` is not angiosarcoma |
| 7 | Resolve extracted entities through the linker | your Ex3 code becomes the referee |
| 8 | Feed the failure back to the model | does it correct or dig in? |
| 9 | Assemble the evidence table and export your prompt log | something you could act on |

There are **seven tasks**, numbered in the order the notebook reaches them. Each one is something you
explain to the class when you present — not a form you fill in. The last one is the habit that makes
the rest reproducible: keeping a **prompt log**.

### Your assignment is different from everyone else's

Your matriculation number's **last digit** selects both a **topic** and an **ontology**. Part 2 reads
the digit and prints what you have been given. Two students with different last digits are working on
different diseases and different vocabularies, so the answers cannot be copied and the comparison in
Task 6 is genuinely informative.

### What you need

- **Ollama** with `qwen3.5:4b` pulled (3.3 GB) — installed in Part 0.
- Ollama will not run on your machine (usually a pool PC whose NVIDIA driver is too old)?
  Section **1.3b** swaps in a Hugging Face `transformers` backend — same notebook, same tasks.
- Python with `requests` and `pandas`.
- **The ontology for your digit** — 7 MB (DOID) or 53 MB (MONDO). Part 2.3 downloads it if you
  do not already have it.
- A **free NCBI API key** — optional but it triples your rate limit. Part 0.3 explains how.

---

# Part 0 — Setup

Three cells. Each checks what you have and prints the exact command to fix anything missing.

## 0.1 Packages

Run this first. It reports your Python version and prints `sys.executable`, which is the interpreter
your kernel is actually using — worth knowing when you have more than one installed, because a `pip
install` into the wrong one is the most common setup failure there is.

In [1]:
# ============================================================================
# 0.1 PACKAGES
# ============================================================================

import importlib
import sys

print("python:      " + sys.version.split()[0])
print("interpreter: " + sys.executable)
print()

_missing = []
for _name in ("requests", "pandas"):
    try:
        _mod = importlib.import_module(_name)
        print("  %-10s %s" % (_name, getattr(_mod, "__version__", "installed")))
    except ImportError:
        _missing.append(_name)
        print("  %-10s MISSING" % _name)

if not _missing:
    import pandas as pd  # noqa: F401  used from Part 9 onwards

print()
if _missing:
    print("Install with:")
    print()
    print("    " + sys.executable + " -m pip install " + " ".join(_missing))
    print()
    print("Then restart the kernel and re-run this cell.")
else:
    print("Both present.")

python:      3.14.7
interpreter: c:\Python\3.14\python.exe

  requests   2.34.2
  pandas     3.1.1

Both present.


## 0.2 Ollama and the model

Ollama is a small server that serves model weights over HTTP on port 11434. It is an endpoint, not
a chat app — the course app uses the same one.

If you have not installed it:

| System | Command |
|---|---|
| **Windows** | `winget install Ollama.Ollama` |
| **macOS** | `brew install ollama` |
| **Linux** | `curl -fsSL https://ollama.com/install.sh \| sh` |

Then **open a new terminal** so `PATH` updates. The cell below finds the executable, starts the
server if it is not running, and pulls the model if it is missing.

**Two failures this cell already knows about.** On the pool PCs an Ollama server may already be
running on port 11434 that you are not allowed to stop — and that server's GPU runner may crash
on every call with `the provided PTX was compiled with an unsupported toolchain`, because the
CUDA code Ollama ships is newer than the installed NVIDIA driver. The cell never tries to stop
anything: it tests the server with a real generation, and when that fails it starts **your own**
server on the next free port, forced onto the CPU (`CUDA_VISIBLE_DEVICES=-1`, `OLLAMA_VULKAN=0`).
CPU-only is slower and entirely adequate for this exercise.


In [3]:
# ============================================================================
# 0.2 OLLAMA + MODEL
#
# Two things can go wrong on a shared machine, and this cell handles both:
#
#  1. a server is already running on port 11434 and you are not allowed to
#     stop it. We never try. We use it if it works; otherwise we start our
#     OWN server on the next free port;
#  2. the server answers, but every generate call fails with HTTP 500
#     because its GPU runner crashed - on the pool PCs the NVIDIA driver is
#     older than the CUDA code Ollama ships ("the provided PTX was compiled
#     with an unsupported toolchain"). We detect that with a real generation
#     and fall back to a CPU-only server.
# ============================================================================

import json
import os
import platform
import re
import shutil
import socket
import subprocess
import time

import requests

OLLAMA_URL = "http://127.0.0.1:11434"
MODEL = "qwen3.5:4b"


def _tags(url, timeout=4):
    """Model names on a server, or None if nothing answers there."""
    try:
        return [m["name"] for m in
                requests.get(url + "/api/tags", timeout=timeout).json()["models"]]
    except Exception:
        return None


def _can_generate(url, timeout=180):
    """True only if the server actually produces text for MODEL.

    A server whose GPU runner crashes still answers /api/tags and then
    returns HTTP 500 on every generate. That is exactly the pool-PC
    failure, so a health check is not enough - this does real work.
    """
    try:
        r = requests.post(url + "/api/generate", json={
            "model": MODEL,
            "prompt": "Reply with: OK",
            "stream": False,
            "think": False,
            "options": {"temperature": 0, "seed": 42, "num_predict": 5},
        }, timeout=timeout)
        return r.status_code == 200
    except Exception:
        return False


def _free_port(start=11435):
    """First port at or above `start` with nothing listening on it."""
    for port in range(start, start + 20):
        with socket.socket() as s:
            if s.connect_ex(("127.0.0.1", port)) != 0:
                return port
    return None


def _pull(url):
    """Best-effort pull of MODEL through the given server."""
    env = dict(os.environ, OLLAMA_HOST=url.replace("http://", ""))
    subprocess.run([_exe, "pull", MODEL], env=env)


_exe = shutil.which("ollama") or r"C:\Users\Mital\AppData\Local\Programs\Ollama\ollama.exe"
print("ollama: " + (_exe or "NOT FOUND"))

if not _exe:
    print()
    print("Install it, then open a NEW terminal and re-run this cell:")
    if platform.system() == "Windows":
        print("    winget install Ollama.Ollama")
    elif platform.system() == "Darwin":
        print("    brew install ollama")
    else:
        print("    curl -fsSL https://ollama.com/install.sh | sh")
else:
    _names = _tags(OLLAMA_URL)

    # --- 1. nothing running: start one the ordinary way ---------------------
    if _names is None:
        print("no server on " + OLLAMA_URL + " - starting one ...")
        subprocess.Popen([_exe, "serve"], stdout=subprocess.DEVNULL,
                         stderr=subprocess.DEVNULL)
        for _ in range(20):
            time.sleep(1)
            _names = _tags(OLLAMA_URL)
            if _names is not None:
                break
        print("  " + ("up" if _names is not None else "still not up"))

    # --- 2. the model --------------------------------------------------------
    if _names is not None:
        print("server up. installed: "
              + (", ".join(_names) if _names else "(none)"))
        if not any(MODEL.split(":")[0] in n for n in _names):
            print()
            print("pulling " + MODEL + " (3.3 GB, one time) ...")
            _pull(OLLAMA_URL)
            _names = _tags(OLLAMA_URL) or []

    # --- 3. the test that matters: can this server generate? -----------------
    if _names is not None:
        print()
        print("test generation ...")
        if _can_generate(OLLAMA_URL):
            print("  OK - " + OLLAMA_URL + " produces text.")
        else:
            print("  FAILED - the server answers but cannot generate.")
            print()
            print("  On a lab PC this is the known GPU-driver crash: the CUDA")
            print("  code Ollama ships is newer than the installed driver.")
            print("  You cannot stop a server you do not own, so this cell")
            print("  starts your OWN, on another port, forced onto the CPU.")
            _port = _free_port()
            _url = "http://127.0.0.1:" + str(_port)
            _env = dict(os.environ,
                        OLLAMA_HOST="127.0.0.1:" + str(_port),
                        CUDA_VISIBLE_DEVICES="-1",   # hide every NVIDIA GPU
                        OLLAMA_VULKAN="0",           # and every Vulkan GPU
                        GGML_VK_VISIBLE_DEVICES="-1")
            print("    " + _url)
            subprocess.Popen([_exe, "serve"], env=_env,
                             stdout=subprocess.DEVNULL,
                             stderr=subprocess.DEVNULL)
            _names2 = None
            for _ in range(30):
                time.sleep(1)
                _names2 = _tags(_url)
                if _names2 is not None:
                    break
            if _names2 is not None:
                if not any(MODEL.split(":")[0] in n for n in _names2):
                    print("  pulling " + MODEL + " for the CPU server ...")
                    _pull(_url)
                if _can_generate(_url, timeout=300):
                    OLLAMA_URL = _url
                    print("  OK - using the CPU-only server on " + OLLAMA_URL)
                    print("  Slower than a GPU, and fine for this exercise.")
                else:
                    print("  the CPU-only server did not work either.")
            if OLLAMA_URL == "http://127.0.0.1:11434":
                print()
                print("  Could not get a working server. Two options:")
                print("    - tell your instructor, quoting the text above;")
                print("    - use the transformers fallback in section 1.3b.")
    else:
        print()
        print("could not reach " + OLLAMA_URL + " - start it yourself with:")
        print("    ollama serve")
        print("(or use the transformers fallback in section 1.3b)")

    # --- 4. version check ----------------------------------------------------
    # "think": false was added in Ollama 0.5. On anything older the field
    # is ignored, the reasoning model fills its token cap thinking, and
    # `response` comes back empty. That failure looks like a model bug and
    # is not one, so it is worth catching here rather than in Part 4.
    # Point the client at the endpoint we settled on, and read both streams:
    # some builds print the version to stderr and connection warnings to
    # stdout, so stdout alone can miss it.
    try:
        _vres = subprocess.run([_exe, "--version"], capture_output=True,
                               text=True, timeout=15,
                               env=dict(os.environ,
                                        OLLAMA_HOST=OLLAMA_URL.replace("http://", "")))
        _vtext = _vres.stdout + " " + _vres.stderr
    except Exception:
        _vtext = ""

    _num = re.search(r"(\d+)\.(\d+)(?:\.(\d+))?", _vtext)
    print()
    print("ollama version: " + (_num.group(0) if _num else "could not determine"))
    if _num:
        _maj, _min = int(_num.group(1)), int(_num.group(2))
        if (_maj, _min) < (0, 5):
            print()
            print("  WARNING: you need Ollama 0.5 or newer for the")
            print('  "think": false option this notebook relies on.')
            print("  On an older version every call returns an empty answer")
            print("  and Part 1.3 will look like the model is broken.")
            print("  Upgrade:  https://ollama.com/download")
    else:
        print("  (could not parse a version number - if Part 1.3 prints")
        print("   empty answers, upgrade Ollama to 0.5 or newer)")

    print()
    print("endpoint this notebook will use: " + OLLAMA_URL)

ollama: C:\Users\pakdamanm1\AppData\Local\Programs\Ollama\ollama.EXE
server up. installed: qwen3.5:4b

test generation ...
  FAILED - the server answers but cannot generate.

  On a lab PC this is the known GPU-driver crash: the CUDA
  code Ollama ships is newer than the installed driver.
  You cannot stop a server you do not own, so this cell
  starts your OWN, on another port, forced onto the CPU.
    http://127.0.0.1:11436
  OK - using the CPU-only server on http://127.0.0.1:11436
  Slower than a GPU, and fine for this exercise.

ollama version: 0.40.17

endpoint this notebook will use: http://127.0.0.1:11436


## 0.3 NCBI API key (optional)

NCBI allows 3 requests/second anonymously and **10/second** with a free key. Part 3 fetches
abstracts, so a key makes it noticeably quicker.

1. <https://www.ncbi.nlm.nih.gov/account/> → create an account
2. **Account Settings** → **API Key Management** → **Create an API Key**

Set it as an environment variable before launching Jupyter so you never paste a secret into a
notebook you might share. Everything works without one.

In [4]:
# ============================================================================
# 0.3 API KEY + A READINESS CHECK
# ============================================================================

NCBI_API_KEY = os.environ.get("NCBI_API_KEY", "")
PAUSE = 0.11 if NCBI_API_KEY else 0.34

print("API key: " + ("found (" + str(len(NCBI_API_KEY)) + " chars)" if NCBI_API_KEY
                     else "not set - fine, just slower"))
print()

_ok = True
try:
    import pandas  # noqa: F401
    print("[OK]   pandas")
except ImportError:
    _ok = False
    print("[FAIL] pandas")
try:
    _names = [m["name"] for m in
              requests.get(OLLAMA_URL + "/api/tags", timeout=4).json()["models"]]
    print("[OK]   ollama server")
    if any("qwen3.5" in n for n in _names):
        print("[OK]   " + MODEL)
    else:
        _ok = False
        print("[FAIL] " + MODEL + " missing - run cell 0.2")
except Exception:
    _ok = False
    print("[FAIL] ollama server - re-run cell 0.2")

print()
print("Ready." if _ok else "Fix the FAIL lines above before continuing.")

API key: not set - fine, just slower

[OK]   pandas
[OK]   ollama server
[OK]   qwen3.5:4b

Ready.


---

# Part 1 — A model you can measure

## 1.1 Why local, in one paragraph

Privacy is the argument people reach for first, and here it is the one that does not apply: you are
sending `MONDO:0007254` and the words "breast cancer" to a model, and there is no confidentiality
interest in that.

The argument that **does** apply is reproducibility. A hosted model can be silently updated between
Monday and Wednesday. Exercise 3 made you record the `.obo` release date for exactly this reason; a
model needs the same treatment — a named tag, a fixed quantisation, a recorded seed. `qwen3.5:4b` is
a specific 3.3 GB artefact with a digest, and if you write the digest down, anyone can repeat your
run.

## 1.2 The trap that will cost you an hour

`qwen3.5:4b` is a **reasoning model**. By default it writes its chain of thought into a separate
field called `thinking`, and leaves `response` **empty** until the thinking finishes:

```
out_tok=900   done_reason='length'   response=''   thinking='Thinking Process: 1. ...'
```

The model generated 900 tokens, hit your cap mid-thought, and never reached the answer. Nothing
errors — HTTP 200, JSON parses, token counts look healthy, and the answer is an empty string.

The fix is one field, `"think": false`. Measured on the same prompt: **900 tokens and no answer**
with thinking on, **39 tokens and clean JSON in 2.5 s** with it off. The helper below sets it.

## 1.3 One call, and the reproducibility test

Send the identical prompt three times at `temperature: 0` with a fixed seed. If the answers are
identical, you have shown that a model can be made into a deterministic instrument — which is what
makes its output a measurement rather than an opinion.

**If this cell raises `HTTPError: 500`**, the server's GPU runner crashed. Cell 0.2 already
tests for that and starts a CPU-only server — re-run 0.2. If the server belongs to someone
else and cannot be fixed, section 1.3b replaces the backend with `transformers`.

## 1.4 Trying other models — and the hardware ceiling

`qwen3.5:4b` is the default because it fits on a laptop. It is **not** the point of the exercise. The
point is the loop, and the most useful thing you can do with the loop is run it against a different
model and see what changes.

**The rule: stay at 14B parameters or below.** The lab machines and most student laptops cap out
there. A 70B model will either refuse to load or spill onto disk and take twenty minutes per call, and
you will not finish in the session. 14B is the ceiling, not a target — 3B to 9B is the comfortable
range.

Models worth trying, all in the Ollama library:

| Tag | Size | Why you might pick it |
|---|---|---|
| `qwen3.5:4b` | 3.3 GB | the default — a reasoning model, and the one the numbers in this handout come from |
| `qwen3.5:9b` | 6.6 GB | same family, twice the size — does the extra scale fix the formatting failures? |
| `qwen2.5:7b-instruct` | 4.7 GB | **not** a reasoning model — no `thinking` field to get wrong |
| `qwen2.5:14b-instruct` | 9.0 GB | the ceiling. Slowest, and the most likely to follow the format |
| `llama3.1:8b` | 4.9 GB | a different family entirely — different failure modes |
| `phi3:mini` | 2.2 GB | tiny. Useful as a floor: how bad does it get? |
| `gemma4:e4b` | 9.6 GB | mixture-of-experts, so it is fast for its size |

Switching is one line. `ask()` takes `model=` on every call:

```python
TEXT, SECS, INFO = ask(PROMPT, model="qwen2.5:7b-instruct")
```

To change the default for the whole notebook, edit `MODEL` in cell 0.2 and re-run from there. Every
call records the model name in the prompt log, so a mixed-model run is still auditable — but pick one
model for your final numbers, and say which.

**What to look for when you switch.** The interesting result is not "the bigger model is better". It
is *which* of the four failure modes changes:

- does a non-reasoning model (`qwen2.5`) still return empty responses?
- does a bigger model follow the `CLAIM | PMID | QUOTE` format without the template echo?
- does the verbatim rate in Part 5 go up, or just the fluency?
- does a different family fail tier 3 in the *same* way, or a new one?

A model that produces fewer claims but better citations is more useful to you than one that produces
ten fluent unverifiable ones. That is the judgement the exercise is training.

### Where to look things up

| Site | What it is for |
|---|---|
| <https://ollama.com/library> | every model tag, its size, and the exact `ollama pull` command |
| <https://ollama.com/search> | search by name or capability when you know what you want |
| <https://artificialanalysis.ai/> | speed, cost and quality side by side — use it to check a model fits your machine |
| <https://lmarena.ai/> | human preference votes — good for "which reads better", useless for "which is correct" |
| <https://huggingface.co/spaces/open-llm-leaderboard/open_llm_leaderboard> | benchmark scores, with the caveat that benchmarks are not your corpus |

One warning about that last row, and it is the same lesson as Part 5: a leaderboard score is a claim
someone else verified against *their* data. It tells you nothing about whether a model will cite
*your* abstracts correctly. The only benchmark that matters here is the one you run.

In [5]:
# ============================================================================
# 1.3 ONE CALL, AND THE DETERMINISM TEST
# ============================================================================

# Every call goes in here. Part 9.3 exports it; Task 7 asks for it.
PROMPT_LOG = []


def ask(prompt, model=MODEL, num_predict=400, temperature=0.0, seed=42,
        think=False, timeout=600):
    """One non-streaming call. Returns (text, seconds, info dict).

    Every call is appended to PROMPT_LOG, which Part 9.3 exports. Keep the
    log - it is how you show what you actually did, rather than what you
    remember doing.
    """
    started = time.time()
    response = requests.post(OLLAMA_URL + "/api/generate", json={
        "model": model,
        "prompt": prompt,
        "stream": False,
        "think": think,
        "options": {"temperature": temperature, "seed": seed,
                    "num_predict": num_predict},
    }, timeout=timeout)
    elapsed = time.time() - started
    response.raise_for_status()
    body = response.json()
    text = body.get("response", "")
    info = {
        "out_tokens": body.get("eval_count"),
        "done_reason": body.get("done_reason"),
        "thought_chars": len(body.get("thinking") or ""),
    }
    PROMPT_LOG.append({
        "n": len(PROMPT_LOG) + 1,
        "model": model,
        "temperature": temperature,
        "seed": seed,
        "num_predict": num_predict,
        "seconds": round(elapsed, 2),
        "out_tokens": info["out_tokens"],
        "done_reason": info["done_reason"],
        "prompt": prompt,
        "response": text,
    })
    return text, elapsed, info


PROMPT = "In one sentence, define 'xanthelasma'. No preamble."

print("=" * 70)
print("TEMPERATURE 0 - greedy, seeded")
print("=" * 70)
_runs = []
for _i in range(3):
    _t, _s, _info = ask(PROMPT, num_predict=120)
    _runs.append(_t.strip())
    print("  run %d  %5.1fs  %s" % (_i + 1, _s, _t.strip()[:78]))

print()
print("  identical across all three: " + str(len(set(_runs)) == 1))
print("  done_reason: " + str(_info["done_reason"]))
print()
if not any(_runs):
    print("  EMPTY ANSWERS. That is the thinking-model trap from 1.2.")
    print("  ask() already passes think=False, so check your Ollama version:")
    print("    ollama --version      (needs 0.5 or newer)")
elif len(set(_runs)) == 1:
    print("  Stable. Your 40-claim run in Part 4 is a measurement, not a sample.")
else:
    print("  Not stable at temperature 0. Record this - it changes how you")
    print("  report every number in Part 5.")

TEMPERATURE 0 - greedy, seeded
  run 1    3.4s  Xanthelasma is a benign skin condition characterized by the presence of small,
  run 2    2.9s  Xanthelasma is a benign skin condition characterized by the presence of small,
  run 3    2.9s  Xanthelasma is a benign skin condition characterized by the presence of small,

  identical across all three: True
  done_reason: stop

  Stable. Your 40-claim run in Part 4 is a measurement, not a sample.


## 1.3b Plan B — transformers instead of Ollama

Everything in this notebook reaches the model through one function, `ask()`, defined in 1.3. That
version speaks HTTP to an Ollama server. **This section defines a second `ask()` that runs the same
model family directly through Hugging Face `transformers`** — same signature, same `PROMPT_LOG`
entries, same determinism test — so Parts 4 to 9 run unchanged on either backend. Use whichever
works; you do not need both.

When you need it:

- the pool PCs, when Ollama's GPU runner keeps crashing (`the provided PTX was compiled with an
  unsupported toolchain`) and the driver is not yours to update;
- any machine where `pip install` is easier than running a server.

### How to switch

1. Run Part 0 and cell 1.3 as far as they go. On this path the `[FAIL] ollama server` line in 0.3
   is expected — ignore it, and skip 1.5 (it lists Ollama tags, which no longer apply).
2. **The code cell below is commented out on purpose.** Click into it, select all, press `Ctrl+/`
   to uncomment, then run it. The first run downloads the full-precision weights (~8 GB, cached).
3. Continue at Part 2 as normal.

### Any inference you want to try

The cell uses the API from the model card — `AutoProcessor` + `AutoModelForMultimodalLM`. If you
only want to see the model answer once, the documented three-liner works in a scratch cell:

```python
from transformers import pipeline

pipe = pipeline("image-text-to-text", model="Qwen/Qwen3.5-4B")
messages = [{"role": "user", "content": [
    {"type": "text", "text": "In one sentence, define 'xanthelasma'. No preamble."}]}]
print(pipe(text=messages, max_new_tokens=64, do_sample=False))
```

For text-only weights the model card offers `AutoModelForCausalLM` (`Qwen3_5ForCausalLM`); the
`ask()` below needs the multimodal auto class because that is what this checkpoint is. If the
`AutoModelForMultimodalLM` import fails, your `transformers` predates Qwen3.5 support — upgrade it.

### Three honest warnings

- **Not the same artefact.** Ollama's `qwen3.5:4b` is a quantised GGUF (3.3 GB); `Qwen/Qwen3.5-4B`
  is the full-precision checkpoint (~8 GB in bf16, so have the RAM). Same family and version — a
  different model file. The same prompt will not give the same answers on the two backends, which
  is 1.1's "named tag, fixed quantisation" point with teeth. The `model` column of your prompt log
  records which one you ran.
- **Determinism is per-backend.** Greedy decoding on fixed weights repeats on the same machine —
  the re-test below shows it — but cross-backend equality is not promised and should not be assumed.
- **Model switching works differently.** 1.4 switches Ollama tags per call with `model=`. Here the
  weights load once: to compare models, change `HF_MODEL_ID` and re-run the cell.

If the weights do not fit in memory or a call is unusably slow, the smaller fix is Ollama forced
onto the CPU — cell 0.2 does that automatically. Choose one backend and say which you used.

In [6]:
# """1.3b OPTIONAL - transformers INSTEAD OF OLLAMA  (Plan B)
#
# Uncomment this cell (select all, then Ctrl+/) and run it. It replaces the
# ask() from 1.3 wholesale: same signature, same PROMPT_LOG entries, same
# info-dict keys - Parts 4 to 9 do not notice the difference.
#
# One-time packages (Qwen3.5 support reached transformers in Feb 2026, so an
# up-to-date version is required):
#
#     pip install -U transformers accelerate torch
#
# The first run downloads the full-precision weights, about 8 GB (one time,
# cached). If that does not fit in memory, use the Ollama CPU fallback that
# cell 0.2 sets up instead - its quantised copy is four times smaller.
#
# This is the API from the model card: AutoProcessor + AutoModelForMultimodalLM.
# The text-only alternative is AutoModelForCausalLM (Qwen3_5ForCausalLM). A
# three-line pipeline("image-text-to-text", ...) version is in the markdown
# above, if you only want to see the model answer once.
# """
#
# HF_MODEL_ID = "Qwen/Qwen3.5-4B"  # the HF counterpart of the tag in 0.2
#
# import torch
# from transformers import AutoProcessor, AutoModelForMultimodalLM
#
# _dev = "auto" if torch.cuda.is_available() else "cpu"  # CPU if no usable driver
# print("device: " + _dev)
#
# _TF_PROC = AutoProcessor.from_pretrained(HF_MODEL_ID)
# _TF_MODEL = AutoModelForMultimodalLM.from_pretrained(HF_MODEL_ID,
#                                                      device_map=_dev)
# _TF_MODEL.eval()
#
# MODEL = HF_MODEL_ID  # the Part 9 summary prints MODEL - make it honest
#
#
# def ask(prompt, model=HF_MODEL_ID, num_predict=400, temperature=0.0,
#         seed=42, think=False, timeout=None):
#     """Same contract as the Ollama ask() in 1.3; see its docstring.
#
#     Differences to record in your Task 1b answer: `model` is only logged
#     (the weights were fixed when this cell loaded them - to switch, change
#     HF_MODEL_ID); temperature is logged but decoding is greedy, which is
#     what temperature=0 means; and Qwen3.5 has no /no_think switch, so
#     thinking is the chat-template flag enable_thinking - the same knob the
#     Ollama path sends as the "think" body field.
#     """
#     torch.manual_seed(seed)
#     started = time.time()
#
#     _inputs = _TF_PROC.apply_chat_template(
#         [{"role": "user", "content": prompt}],
#         tokenize=True, return_dict=True, return_tensors="pt",
#         add_generation_prompt=True,
#         enable_thinking=think,          # think=False -> direct answer
#     )
#     _inputs = {k: v.to(_TF_MODEL.device) for k, v in _inputs.items()}
#     _in_len = _inputs["input_ids"].shape[1]
#
#     with torch.no_grad():
#         _out = _TF_MODEL.generate(**_inputs, max_new_tokens=num_predict,
#                                   do_sample=False)
#
#     _new = _out[0][_in_len:]
#     _text = _TF_PROC.batch_decode([_new], skip_special_tokens=True)[0]
#
#     _thought = 0
#     if " response" in _text:            # keep the answer, count the rest
#         _head, _, _text = _text.rpartition("</think>")
#         _thought = len(_head)
#
#     elapsed = time.time() - started
#     info = {
#         "out_tokens": int(_new.shape[0]),
#         "done_reason": "length" if _new.shape[0] >= num_predict else "stop",
#         "thought_chars": _thought,
#     }
#     PROMPT_LOG.append({
#         "n": len(PROMPT_LOG) + 1,
#         "model": model,
#         "temperature": temperature,
#         "seed": seed,
#         "num_predict": num_predict,
#         "seconds": round(elapsed, 2),
#         "out_tokens": info["out_tokens"],
#         "done_reason": info["done_reason"],
#         "prompt": prompt,
#         "response": _text,
#     })
#     return _text, elapsed, info
#
#
# print("=" * 70)  # the same determinism test as 1.3, on this backend
# print("TEMPERATURE 0 - greedy, seeded   (transformers backend)")
# print("=" * 70)
# _runs = []
# for _i in range(3):
#     _t, _s, _info = ask(PROMPT, num_predict=120)  # PROMPT is from cell 1.3
#     _runs.append(_t.strip())
#     print("  run %d  %5.1fs  %s" % (_i + 1, _s, _t.strip()[:78]))
#
# print()
# print("  identical across all three: " + str(len(set(_runs)) == 1))
# print("  done_reason: " + str(_info["done_reason"]))

## 1.5 Which models have you got?

Run this before you decide to switch. It lists what is actually installed, flags anything above the
14B ceiling, and prints the exact pull command for the ones you do not have yet.

In [7]:
# ============================================================================
# 1.5 WHAT IS INSTALLED, AND DOES IT FIT
# ============================================================================

# 14B is the ceiling for this course - see 1.4. Raise it only if you know
# your machine can take it, and expect much slower calls.
SIZE_CEILING_B = 14

SUGGESTED = [
    ("qwen3.5:4b", "default - reasoning model"),
    ("qwen3.5:9b", "same family, bigger"),
    ("qwen2.5:7b-instruct", "not a reasoning model"),
    ("qwen2.5:14b-instruct", "the ceiling"),
    ("llama3.1:8b", "different family"),
    ("phi3:mini", "tiny floor"),
]


def size_in_b(name):
    """Rough parameter count in billions, parsed from the tag."""
    _m = re.search(r"(\d+(?:\.\d+)?)\s*[bB]\b", name)
    return float(_m.group(1)) if _m else None


try:
    _installed = {m["name"]: m.get("size") or 0
                  for m in requests.get(OLLAMA_URL + "/api/tags",
                                        timeout=8).json()["models"]}
except Exception:
    _installed = {}

print("=" * 74)
print("INSTALLED MODELS")
print("=" * 74)
if not _installed:
    print("  none reachable - is the server running?")
for _n, _bytes in sorted(_installed.items()):
    _b = size_in_b(_n)
    _flag = ""
    if _b is not None and _b > SIZE_CEILING_B:
        _flag = "  <-- ABOVE THE %dB CEILING" % SIZE_CEILING_B
    # A model tag without an explicit size suffix has no parameter count we
    # can trust, so say so rather than printing a misleading number.
    _sz = "%5.1f GB" % (_bytes / 1e9) if _bytes else "     ?  "
    print("  %-26s %s  %s%s"
          % (_n, _sz, ("~%gB" % _b) if _b is not None else "size unknown", _flag))

print()
print("=" * 74)
print("SUGGESTED FOR THIS EXERCISE")
print("=" * 74)
_missing = []
for _n, _why in SUGGESTED:
    _have = _n in _installed
    print("  %-26s %-26s %s" % (_n, _why, "installed" if _have else "not installed"))
    if not _have:
        _missing.append(_n)

if _missing:
    print()
    print("To pull any of the missing ones (each is a one-time download):")
    for _n in _missing:
        print("    ollama pull " + _n)
    print()
    print("You do NOT need all of them. One extra model is enough to answer")
    print("the question in 1.4 - pick a different size or a different family.")

print()
print("current default: " + MODEL)
_b = size_in_b(MODEL)
if _b is not None and _b > SIZE_CEILING_B:
    print("  WARNING: the default is above the ceiling. Set MODEL in cell 0.2")
    print("  to something at or below %dB before you go further." % SIZE_CEILING_B)

INSTALLED MODELS
  qwen3.5:4b                   3.3 GB  ~4B

SUGGESTED FOR THIS EXERCISE
  qwen3.5:4b                 default - reasoning model  installed
  qwen3.5:9b                 same family, bigger        not installed
  qwen2.5:7b-instruct        not a reasoning model      not installed
  qwen2.5:14b-instruct       the ceiling                not installed
  llama3.1:8b                different family           not installed
  phi3:mini                  tiny floor                 not installed

To pull any of the missing ones (each is a one-time download):
    ollama pull qwen3.5:9b
    ollama pull qwen2.5:7b-instruct
    ollama pull qwen2.5:14b-instruct
    ollama pull llama3.1:8b
    ollama pull phi3:mini

You do NOT need all of them. One extra model is enough to answer
the question in 1.4 - pick a different size or a different family.

current default: qwen3.5:4b


---

# Part 2 — Your assignment, and your referee

## 2.1 The digit table

Your matriculation number's **last digit** gives you a topic and an ontology. Every digit is a
different disease in a different vocabulary, so the work is genuinely individual.

| Digit | Topic | Ontology |
|---|---|---|
| **0** | angiosarcoma | DOID |
| **1** | osteosarcoma | DOID |
| **2** | multiple myeloma | DOID |
| **3** | nephrotic syndrome | DOID |
| **4** | Hashimoto thyroiditis | MONDO |
| **5** | urticaria | MONDO |
| **6** | rickets | MONDO |
| **7** | dental caries | MONDO |
| **8** | cutis laxa | MONDO |
| **9** | xeroderma pigmentosum | MONDO |

All ten topics return between **31 and 39** usable abstracts from the 40 the notebook requests, so none
of them starves. Every one of the ten also resolves to a real identifier in its assigned vocabulary —
that was checked before the exercise was written, because a digit whose topic did not resolve would
make Part 7 collapse.

## 2.2 Why you need the ontology at all

Because it is the thing that decides whether the model's claims are checkable. Part 7 pushes every
entity the model extracts through the linking cascade you built in Exercise 3, and Part 6 checks
whether a mention is the disease itself or something merely *like* it.

Without a vocabulary, "the model said angiosarcoma" is an opinion. With one, it is a row that either
resolves to an identifier or does not.

In [17]:
# ============================================================================
# 2.1 YOUR ASSIGNMENT
# ============================================================================

import re

# --- EDIT THIS: the last digit of your matriculation number ------------------
MY_DIGIT = 2
# ----------------------------------------------------------------------------

ASSIGNMENTS = {
    0: ("angiosarcoma", "doid", "DOID"),
    1: ("osteosarcoma", "doid", "DOID"),
    2: ("multiple myeloma", "doid", "DOID"),
    3: ("nephrotic syndrome", "doid", "DOID"),
    4: ("Hashimoto thyroiditis", "mondo", "MONDO"),
    5: ("urticaria", "mondo", "MONDO"),
    6: ("rickets", "mondo", "MONDO"),
    7: ("dental caries", "mondo", "MONDO"),
    8: ("cutis laxa", "mondo", "MONDO"),
    9: ("xeroderma pigmentosum", "mondo", "MONDO"),
}

assert 0 <= MY_DIGIT <= 9, "MY_DIGIT must be a single digit 0-9"
MY_TOPIC, MY_VOCAB, MY_PREFIX = ASSIGNMENTS[MY_DIGIT]

print("=" * 62)
print("YOUR ASSIGNMENT")
print("=" * 62)
print("  matriculation last digit: " + str(MY_DIGIT))
print("  topic:                    " + MY_TOPIC)
print("  ontology:                 " + MY_VOCAB.upper() + "  (" + MY_PREFIX + ")")
print("=" * 62)
print()
print("Everyone with a different last digit has a different topic AND a")
print("different vocabulary, so your numbers are not comparable by copying -")
print("they are comparable in Task 6, which is the point.")

YOUR ASSIGNMENT
  matriculation last digit: 2
  topic:                    multiple myeloma
  ontology:                 DOID  (DOID)

Everyone with a different last digit has a different topic AND a
different vocabulary, so your numbers are not comparable by copying -
they are comparable in Task 6, which is the point.


## 2.3 Load the ontology

Three things come out of the `.obo` file: the terms, the `is_a` graph, and the label index the linker
needs.

**You do not have to download anything by hand.** If `cache/<your-vocab>.obo` is not there, this cell
fetches it — about 7 MB for DOID and 53 MB for MONDO, so give it a minute on a slow connection and
do not interrupt it. If the file is already there, it is reused and the cell takes a second.

Two details are worth noticing, because both are about trusting your own data:

- it writes to `<file>.part` and only moves the file into place after checking it, so a connection
  that dies halfway cannot leave you with half an ontology that parses without complaint and quietly
  wrong numbers everywhere downstream;
- it prints the `data-version:` line from the header. That is the release you are working against, and
  it belongs in your prompt log next to the model tag — the same way Exercise 3 made you record the
  release of your `.obo`.

If the download fails — a corporate proxy, no network, a machine that blocks `purl.obolibrary.org`
— the cell prints the URL and the exact path to save it to. Fetch it in a browser, put it there,
re-run the cell.


In [18]:
# ============================================================================
# 2.3 LOAD THE ONTOLOGY
#
# You need exactly one .obo file: the vocabulary for your digit. If it is not
# in cache/ already, this cell downloads it - about 7 MB for DOID, 53 MB for
# MONDO. Nothing else is fetched.
# ============================================================================

import io

CACHE_DIR = "cache"
OBO_URL = "https://purl.obolibrary.org/obo/%s.obo"
OBO_PATH = os.path.join(CACHE_DIR, MY_VOCAB + ".obo")

# A download that dies halfway is worse than no file at all: it parses without
# complaint and every number downstream is quietly wrong. So a file is only
# accepted if it has a real OBO header and is big enough to be complete, and a
# download is only accepted once it has been checked and moved into place in one
# step. The stanza count in _load() is the final check.
# Measured on the releases this exercise was written against: doid.obo is
# 7.3 MB / 14939 stanzas, mondo.obo is 53.4 MB / 63391 stanzas. The floors sit
# near 70% of that - low enough that a new release will not trip them, high
# enough that a transfer cut off part-way through will.
MIN_BYTES = {"doid": 5 << 20, "mondo": 37 << 20}
MIN_TERMS = {"doid": 10000, "mondo": 44000}


def parse_obo(path):
    """{id: {name, def, parents, synonyms, obsolete}} for every [Term]."""
    terms, cur = {}, None
    with io.open(path, encoding="utf-8", errors="replace") as fh:
        for line in fh:
            line = line.rstrip("\n")
            if line.startswith("["):
                if cur and cur.get("id"):
                    terms[cur["id"]] = cur
                cur = ({"id": None, "name": None, "def": None, "parents": [],
                        "synonyms": [], "obsolete": False}
                       if line == "[Term]" else None)
                continue
            if cur is None:
                continue
            if line.startswith("id: "):
                cur["id"] = line[4:].strip()
            elif line.startswith("name: "):
                cur["name"] = line[6:].strip()
            elif line.startswith("def: "):
                cur["def"] = line[5:].strip()
            elif line.startswith("is_a: "):
                cur["parents"].append(line[6:].strip().split(" ! ")[0])
            elif line.startswith("synonym: "):
                _m = re.match(r'synonym: "(.*?)"', line)
                if _m:
                    cur["synonyms"].append(_m.group(1))
            elif line.startswith("is_obsolete: true"):
                cur["obsolete"] = True
    if cur and cur.get("id"):
        terms[cur["id"]] = cur
    return terms


def _looks_like_obo(path, vocab=None):
    """Cheap check that a file is a complete .obo and not an error page.

    Two things only: the OBO header line, and a size floor for the vocabulary.
    Do NOT look for "[Term]" in the first few KB - MONDO's header is 7 KB long,
    so its first stanza is well past any reasonable head read.
    """
    try:
        floor = MIN_BYTES.get(vocab or MY_VOCAB, 10000)
        if os.path.getsize(path) < floor:
            return False
        with io.open(path, encoding="utf-8", errors="replace") as fh:
            head = fh.read(4096)
        return "format-version:" in head
    except OSError:
        return False


def _release_of(path):
    """The release string the file carries - this is what makes it citable."""
    with io.open(path, encoding="utf-8", errors="replace") as fh:
        for line in fh:
            if line.startswith("["):
                break
            if line.startswith("data-version:"):
                return line.split(":", 1)[1].strip()
            if line.startswith("date:"):
                return line.split(":", 1)[1].strip()
    return "no release recorded in the header"


def _download(vocab, path):
    """Download to <path>.part, check it, then move it into place."""
    url = OBO_URL % vocab
    part = path + ".part"
    for attempt in (1, 2):
        try:
            print("  GET " + url + ("" if attempt == 1 else "   (retry)"))
            got = 0
            with requests.get(url, stream=True, timeout=600) as r:
                r.raise_for_status()
                # If the server tells us how big the file is, hold it to that.
                # Only when the body is sent as-is: requests decompresses gzip
                # for us, so for a compressed response Content-Length counts the
                # compressed bytes and comparing it to what we wrote is nonsense.
                expected = None
                if not r.headers.get("Content-Encoding") \
                        and not r.headers.get("Transfer-Encoding"):
                    _cl = r.headers.get("Content-Length", "")
                    if _cl.isdigit():
                        expected = int(_cl)
                with open(part, "wb") as fh:
                    for chunk in r.iter_content(1 << 20):
                        fh.write(chunk)
                        got += len(chunk)
                        if got % (10 << 20) < len(chunk):
                            print("    " + str(got >> 20) + " MB ...")
            if expected is not None and int(expected) != got:
                raise ValueError("short transfer: " + str(got) + " of "
                                 + str(expected) + " bytes")
            if not _looks_like_obo(part, vocab):
                raise ValueError(
                    "the download is not a complete OBO ("
                    + format(os.path.getsize(part) / 1e6, ".1f")
                    + " MB) - truncated, or an error page")
            os.replace(part, path)
            return
        except Exception as _exc:
            print("    failed: " + str(_exc))
            if os.path.exists(part):
                os.remove(part)
            if attempt == 2:
                raise SystemExit(
                    "\nCould not download " + vocab + ".obo.\n\n"
                    "Either fix the network and re-run this cell, or download it\n"
                    "in a browser from\n\n    " + url + "\n\n"
                    "and save it as\n\n    " + os.path.abspath(path) + "\n\n"
                    "then re-run this cell. It is plain text - you can open it\n"
                    "and read it.")


def ensure_obo(vocab):
    """Return the local path of <vocab>.obo, downloading it if it is missing."""
    os.makedirs(CACHE_DIR, exist_ok=True)
    path = os.path.join(CACHE_DIR, vocab + ".obo")
    if _looks_like_obo(path, vocab):
        print("cached   " + path + "  ("
              + format(os.path.getsize(path) / 1e6, ".1f") + " MB)")
        return path
    if os.path.exists(path):
        print("the file at " + path + " is not a usable OBO - replacing it")
    print("downloading " + (OBO_URL % vocab) + " ...")
    _download(vocab, path)
    print("  saved  " + format(os.path.getsize(path) / 1e6, ".1f") + " MB")
    return path


def _load(vocab):
    """Path + parsed terms, downloading or repairing the file if needed."""
    path = ensure_obo(vocab)
    terms = parse_obo(path)
    floor = MIN_TERMS.get(vocab)
    if floor and len(terms) < floor:
        print("  only " + str(len(terms)) + " stanzas in " + path
              + " - that is a truncated file, not a release")
        os.remove(path)
        print("  downloading a fresh copy ...")
        _download(vocab, path)
        terms = parse_obo(path)
        if len(terms) < floor:
            raise SystemExit(
                "\n" + path + " still has only " + str(len(terms)) + " stanzas\n"
                "after a fresh download. A complete " + vocab + ".obo has at\n"
                "least " + str(floor) + ". Something upstream is wrong - tell\n"
                "your instructor, and say which release the header names.")
        print("  repaired: " + str(len(terms)) + " stanzas")
    return path, terms


OBO_PATH, ALL_TERMS = _load(MY_VOCAB)
print("release  " + _release_of(OBO_PATH))

OWN_TERMS = {k: v for k, v in ALL_TERMS.items()
             if k.upper().startswith(MY_PREFIX) and not v["obsolete"] and v["name"]}

LABEL_TO_IDS = {}
for _tid, _t in OWN_TERMS.items():
    LABEL_TO_IDS.setdefault(_t["name"].lower(), []).append(_tid)

print()
print("stanzas in file:      " + str(len(ALL_TERMS)))
print("own, usable terms:    " + str(len(OWN_TERMS)))
print("distinct labels:      " + str(len(LABEL_TO_IDS)))


downloading https://purl.obolibrary.org/obo/doid.obo ...
  GET https://purl.obolibrary.org/obo/doid.obo
  saved  7.3 MB
release  releases/2026-10-29/doid.obo

stanzas in file:      14939
own, usable terms:    12419
distinct labels:      12419


## 2.4 The referee: your Exercise 3 linker

This is the code you wrote in Exercise 3, reduced to its core. It takes a **mention** — a string
lifted from a paper — and returns either a verified identifier or an honest refusal.

The cascade is the same one: exact label, then synonym, then a fuzzy match that is *never allowed to
become a link*. Part 7 uses it to decide whether the model's extracted entities are real.

If you did Exercise 3, you will recognise `resolve_mention`. If you did not, it does not matter —
read the docstring, it says exactly what it promises.

In [19]:
# ============================================================================
# 2.4 THE LINKER - your Exercise 3 cascade, as a function
# ============================================================================

import difflib

# Synonym -> ids, so the linker can match on synonyms as well as labels.
SYNONYM_TO_IDS = {}
for _tid, _t in OWN_TERMS.items():
    for _syn in _t["synonyms"]:
        SYNONYM_TO_IDS.setdefault(_syn.lower(), []).append(_tid)

FUZZY_THRESHOLD = 0.90


def normalise(text):
    """Lowercase, punctuation to space, collapse whitespace."""
    text = text.lower()
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def resolve_mention(mention, allow_fuzzy=False):
    """Resolve a string to an ontology id.

    Returns (status, id_or_None, evidence) where status is one of:
      'linked_label'   exact label match
      'linked_synonym' exact synonym match
      'ambiguous'      several ids share the label - refuse to choose
      'fuzzy'          a close match exists but is NOT a link
      'unlinked'       nothing found

    Mirrors the Exercise 3 cascade: fuzzy matches are reported, never linked.
    """
    key = normalise(mention)
    if not key:
        return "unlinked", None, "empty mention"

    ids = LABEL_TO_IDS.get(key)
    if ids:
        if len(ids) == 1:
            return "linked_label", ids[0], "exact label"
        return "ambiguous", None, "label shared by " + str(len(ids)) + " terms"

    ids = SYNONYM_TO_IDS.get(key)
    if ids:
        if len(ids) == 1:
            return "linked_synonym", ids[0], "exact synonym"
        return "ambiguous", None, "synonym shared by " + str(len(ids)) + " terms"

    close = difflib.get_close_matches(key, list(LABEL_TO_IDS), n=1,
                                      cutoff=FUZZY_THRESHOLD)
    if close:
        return ("fuzzy", None,
                "closest '" + close[0] + "' at "
                + format(difflib.SequenceMatcher(None, key, close[0]).ratio(), ".2f"))
    return "unlinked", None, "no candidate"


# Sanity check: the topic itself should resolve in your own vocabulary.
_status, _tid, _why = resolve_mention(MY_TOPIC)
print("sanity check - your topic as a mention:")
print("  " + repr(MY_TOPIC) + "  ->  " + _status
      + "  " + str(_tid or "") + "  (" + _why + ")")
print()
if _status.startswith("linked"):
    print("  The referee knows your topic: " + OWN_TERMS[_tid]["name"])
else:
    print("  Note: your topic does not resolve by exact label in "
          + MY_VOCAB.upper() + ".")
    print("  That is itself worth mentioning - and Part 7 will show what the")
    print("  linker does with the model's variants of the same word.")

sanity check - your topic as a mention:
  'multiple myeloma'  ->  linked_label  DOID:9538  (exact label)

  The referee knows your topic: multiple myeloma


---

# Part 3 — Build the corpus

## 3.1 A Boolean query is a decision procedure

Not "search for these words" — for every document in a corpus of 37 million, it answers yes or no.
Getting it right means controlling both errors: too broad and you retrieve 60,000 papers you never
read; too narrow and you retrieve twelve and miss the one that mattered.

You have **one topic**, so the query is small. That is deliberate: the interesting failure in this
exercise is not query construction, it is what happens to the citations afterwards.

## 3.2 The field that surprises everyone

PubMed does not index free text — it indexes with MeSH, and MeSH descriptor names are frequently not
the ontology label. Measured directly:

| Query | Hits |
|---|---|
| `"Hashimoto thyroiditis"[MeSH Terms]` | **0** |
| `"Hashimoto Disease"[MeSH Terms]` | 5,421 |
| `"Hashimoto thyroiditis"[tiab]` | 7,551 |

There is no MeSH descriptor called "Hashimoto thyroiditis". The string is valid, PubMed returns zero,
and nothing warns you.

This is Exercise 3's lesson in a new place: **the label is not the identifier, and the identifier is
not the name.** You spent that exercise learning that `MONDO:0007699` and the string
"Hashimoto thyroiditis" are different things. Here the gap sits between a MeSH ID (`D006394`), a MeSH
descriptor name (`"Hemangiosarcoma"`), and the ontology label (`angiosarcoma`).

We use `[tiab]` for that reason — and Part 5 gives you a way to check it actually did what you asked.

In [20]:
# ============================================================================
# 3.1 BUILD AND RUN THE QUERY
# ============================================================================

EUTILS = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"
TARGET_ABSTRACTS = 40        # how many papers to collect
BATCH = 100


def eutils(endpoint, **params):
    q = {"tool": "biodb-ex4", "email": "student@example.org"}
    if NCBI_API_KEY:
        q["api_key"] = NCBI_API_KEY
    q.update(params)
    r = requests.get(EUTILS + "/" + endpoint, params=q, timeout=90)
    if r.status_code == 429:
        raise RuntimeError("rate limited - wait 10 s, or set NCBI_API_KEY")
    r.raise_for_status()
    return r


QUERY = '"' + MY_TOPIC + '"[tiab] AND hasabstract[text]'
print("query: " + QUERY)

_response = eutils("esearch.fcgi", db="pubmed", term=QUERY,
                   retmax=TARGET_ABSTRACTS, retmode="json", sort="relevance")
_result = _response.json()["esearchresult"]
TOTAL_HITS = int(_result["count"])
PMIDS = _result.get("idlist", [])

print("  total hits in PubMed: " + str(TOTAL_HITS))
print("  collected:            " + str(len(PMIDS)))
print()
if len(PMIDS) < 10:
    print("Fewer than 10 abstracts - your topic may be too narrow. Tell your")
    print("instructor rather than continuing; the exercise needs real text.")
elif TOTAL_HITS > 20000:
    print("This is a broad topic. Your 40 papers are the most RELEVANT, not a")
    print("random sample - say so in Task 1, because it changes what the")
    print("precision number in Part 5 means.")

query: "multiple myeloma"[tiab] AND hasabstract[text]
  total hits in PubMed: 48028
  collected:            40

This is a broad topic. Your 40 papers are the most RELEVANT, not a
random sample - say so in Task 1, because it changes what the
precision number in Part 5 means.


## 3.3 The filter that is not optional

The query above is not just `"your topic"[tiab]`. It ends with:

```
AND hasabstract[text]
```

That clause exists because of a trap that silently empties the exercise. Relevance ranking on a bare
`[tiab]` query returns the **most-cited, most-matched** records, and for many disease names those are
1960s titles, editorials, letters and conference abstracts — records that have a title and **no
abstract at all**. Measured across all ten digits, a bare query gave:

| Digit | Topic | Records returned | With a usable abstract |
|---|---|---|---|
| 7 | dental caries | 40 | **0** |
| 3 | nephrotic syndrome | 40 | **1** |
| 2 | multiple myeloma | 40 | **4** |
| 9 | xeroderma pigmentosum | 40 | **4** |

Forty records, zero abstracts, no error. With `hasabstract[text]`, the same ten topics return
**31–39 usable abstracts each**. Same corpus, same ranking, one clause.

That is the third version of the same lesson in this course: a query that returns nothing looks
exactly like a topic nobody has studied. In Exercise 3 it was a MeSH descriptor name that did not
exist; here it is a query that matches records you cannot read.

## 3.4 Fetch the abstracts

`efetch` with `retmode=xml` gives structured `AbstractText` elements — Background, Methods, Results
stay separate instead of being flattened into one blob with headings. That matters when you check a
quote later: you want the sentence as the authors wrote it.

The filter does most of the work, but a few records still come back with a stub or no abstract. Those
are dropped, and the count is reported — so you can see the difference between what you asked for and
what you got.

In [21]:
# ============================================================================
# 3.2 FETCH AND PARSE
# ============================================================================

import xml.etree.ElementTree as ET

PAPERS = []
for _i in range(0, len(PMIDS), BATCH):
    _chunk = PMIDS[_i:_i + BATCH]
    _r = eutils("efetch.fcgi", db="pubmed", id=",".join(_chunk),
                rettype="abstract", retmode="xml")
    _root = ET.fromstring(_r.text)
    for _art in _root.iter("PubmedArticle"):
        _pmid = _art.findtext(".//PMID") or ""
        _title = (_art.findtext(".//ArticleTitle") or "").strip()
        _parts = [_n.text or "" for _n in _art.iter("AbstractText")]
        _abstract = " ".join(_parts).strip()
        PAPERS.append({
            "pmid": _pmid,
            "title": _title,
            "abstract": _abstract,
            "journal": _art.findtext(".//Journal/Title") or "",
            "year": _art.findtext(".//PubDate/Year") or "",
        })
    time.sleep(PAUSE)

USABLE = [p for p in PAPERS if len(p["abstract"]) > 150]
print("fetched:        " + str(len(PAPERS)))
print("with abstract:  " + str(len(USABLE)))
print("mean length:    " + str(int(sum(len(p["abstract"]) for p in USABLE)
                                 / max(len(USABLE), 1))) + " chars")
print("journals:       " + str(len({p["journal"] for p in USABLE})))
print()
if len(USABLE) < 10:
    print("Too few usable abstracts. Increase TARGET_ABSTRACTS above and re-run.")
else:
    print("Corpus ready. Example:")
    print("  PMID " + USABLE[0]["pmid"] + "  " + USABLE[0]["title"][:70])

fetched:        39
with abstract:  35
mean length:    452 chars
journals:       28

Corpus ready. Example:
  PMID 40743872  Flow cytometry in the diagnosis and prognosis of multiple myeloma.


> **TASK 1 — your assignment and your corpus.** Open by telling the class your digit, your topic, and
> your ontology. Then describe the corpus you ended up with: total PubMed hits for your query, how
> many you collected, how many had usable abstracts, and the mean abstract length. Say out loud
> whether your 40 papers are a **random sample** or the **most relevant** ones — and what that does to
> any precision claim you make later. This is the same provenance discipline Exercise 3's Task 1 asked
> for, applied to a corpus instead of a file.

> **TASK 1b — the model you used, and one you tried.** Name the exact model tag from cell 0.2 and say
> whether it is a reasoning model. If you ran the loop against a second model from 1.4, say which one
> and what changed: did the formatting hold, did the verbatim rate move, did the same claims fail tier 3
> for the same reason? A model that produces fewer but checkable claims is more useful than one that
> produces ten fluent unverifiable ones — and if you only ran the default, say that plainly rather
> than implying you compared.

---

# Part 4 — The same question, twice

## 4.1 The experiment

You will ask the model the same thing twice:

- **Condition A — memory.** No context. "What is known about your topic? Cite PMIDs."
- **Condition B — grounded.** The same question, plus the abstracts from your corpus.

Nothing else changes: same model, same temperature, same seed, same number of claims. Only the
context differs. That is what makes the comparison a controlled one.

## 4.2 What to expect, and why it matters

On the reference run for this exercise (DOID, 20 abstracts, three topics) the two conditions differed
completely:

| | PMIDs cited | Existed in the corpus | Quotes verbatim |
|---|---|---|---|
| **A — memory** | 15 | **0** | — |
| **B — grounded** | 10 | **10** | **10 of 10** |

That is the whole lesson in one table, and it is not "the model is bad". It is that **a model without
your documents is answering from training data**, and training data has no obligation to your
release, your query, or your topic.

## 4.3 Why the failure is more dangerous than it looks

The obvious fear is that the model invents identifier numbers. The real problem is worse. When the
reference run's cited PMIDs were checked against PubMed, **11 of 12 existed**:

| PMID | What it actually is |
|---|---|
| 30936654 | Growth performances, carcass traits, meat quality — **pig nutrition** |
| 28715632 | All-graphene oxide flexible **supercapacitors** |
| 31234567 | Spatial distribution of **moisture in granular** material |
| 15748755 | **Badger removal** and tuberculosis in cattle |
| 19832485 | *does not exist* |

Eleven plausible, resolving, real identifiers attached to claims about cancer. A made-up number is
easy to dismiss. `PMID:30936654` looks authoritative and is a paper about pig carcasses. **That is the
failure this exercise is about**, and Part 5 gives you three ways to catch it.

In [23]:
# ============================================================================
# 4.1 CONDITION A - MEMORY (no context)
# ============================================================================

N_CLAIMS = 5

PROMPT_MEMORY = (
    "You are a biomedical researcher.\n"
    "Summarise what is known about {topic} in at most " + str(N_CLAIMS) + " short factual claims.\n"
    "For each claim, cite the PubMed PMID that supports it.\n"
    "Format each line exactly as: CLAIM | PMID:12345678\n"
    "Do not add commentary, headings or preamble.\n"
)

print("=" * 74)
print("CONDITION A - MEMORY.  Topic: " + MY_TOPIC)
print("=" * 74)
A_TEXT, A_SECS, A_INFO = ask(PROMPT_MEMORY.replace("{topic}", MY_TOPIC),
                             num_predict=500)
print("(" + format(A_SECS, ".1f") + " s, " + str(A_INFO["out_tokens"])
      + " tokens, done=" + str(A_INFO["done_reason"]) + ")")
print()
print(A_TEXT.strip())
print()
print("=" * 74)
print("Read that. It is fluent, specific, and cites five papers. You have no")
print("way to check any of it from here - and that is the point of Part 5.")
print("=" * 74)

CONDITION A - MEMORY.  Topic: multiple myeloma
(17.4 s, 191 tokens, done=stop)

Multiple myeloma is a hematologic malignancy characterized by the neoplastic proliferation of plasma cells in the bone marrow. | PMID:20933314
The disease typically presents with CRAB features: hypercalcemia, renal insufficiency, anemia, and lytic bone lesions. | PMID:15768445
Diagnosis requires a clonal population of malignant plasma cells exceeding 10% in the bone marrow or an extramedullary mass. | PMID:23942834
Treatment strategies have evolved to include proteasome inhibitors, immunomodulatory drugs, and monoclonal antibodies targeting CD38 and SLAMF7. | PMID:35621089
Prognosis is generally poor without treatment, but outcomes have improved significantly with the advent of novel targeted therapies. | PMID:34567890

Read that. It is fluent, specific, and cites five papers. You have no
way to check any of it from here - and that is the point of Part 5.


In [24]:
# ============================================================================
# 4.2 CONDITION B - GROUNDED (the same question, plus your abstracts)
# ============================================================================

# Give the model a fixed slice of the corpus so the experiment is repeatable.
CONTEXT_N = 8
CONTEXT_PAPERS = USABLE[:CONTEXT_N]
CONTEXT = "\n\n".join(
    "PMID:%s\nTITLE: %s\nABSTRACT: %s"
    % (p["pmid"], p["title"], p["abstract"][:1200])
    for p in CONTEXT_PAPERS
)

PROMPT_GROUNDED = (
    "You are a biomedical researcher. Below are abstracts from PubMed.\n"
    "Using ONLY these abstracts, state at most " + str(N_CLAIMS)
    + " factual claims about {topic}.\n"
    "For each claim, cite the PMID it came from and quote the supporting sentence.\n"
    "Format each line exactly as:\n"
    "CLAIM: <the claim> | PMID:12345678 | QUOTE: <a verbatim sentence from that abstract>\n"
    "If the abstracts do not support a claim, do not make it.\n"
    "Do not add commentary.\n"
    "\nABSTRACTS:\n{context}\n"
)

B_PROMPT = (PROMPT_GROUNDED.replace("{topic}", MY_TOPIC)
            .replace("{context}", CONTEXT))

print("=" * 74)
print("CONDITION B - GROUNDED.  " + str(len(CONTEXT_PAPERS))
      + " abstracts given.")
print("=" * 74)
B_TEXT, B_SECS, B_INFO = ask(B_PROMPT, num_predict=900)
print("(" + format(B_SECS, ".1f") + " s, " + str(B_INFO["out_tokens"])
      + " tokens, done=" + str(B_INFO["done_reason"]) + ")")
print()
print(B_TEXT.strip())
print()
if B_INFO["done_reason"] == "length":
    print("NOTE: done_reason='length' means the answer was cut off at the")
    print("num_predict cap. The last claim in your output is probably missing")
    print("words, and later claims are missing entirely.")
    print()
    print("Fix: raise num_predict in the ask() call above (try 1400) and")
    print("re-run this cell. Do not analyse a truncated answer - a quote that")
    print("stops mid-sentence will fail tier 2 for a reason that has nothing")
    print("to do with the model's honesty.")

CONDITION B - GROUNDED.  8 abstracts given.
(49.4 s, 381 tokens, done=stop)

CLAIM: Multiple myeloma is a disease characterized by abnormal clonal plasma cells that produce paraprotein. | PMID:36540606 | QUOTE: Myeloma is a disease characterized by abnormal clonal plasma cells that produces paraprotein.
CLAIM: Multiple myeloma is a plasma cell dyscrasia characterized by abnormal bone marrow clonal plasma cells, histological confirmation of plasmacytoma, monoclonal protein in serum or urine, and evidence of end-organ damage. | PMID:34659990 | QUOTE: Multiple myeloma is a plasma cell dyscrasia characterized by abnormal bone marrow clonal plasma cells, histological confirmation of plasmacytoma, monoclonal protein in serum or urine, and evidence of end-organ damage.
CLAIM: Organ involvement in multiple myeloma manifests as CRAB (hyperCalcemia, Renal insufficiency, Anemia, lytic Bone lesions). | PMID:34659990 | QUOTE: Organ involvement in multiple myeloma manifests as CRAB (hyperCalcemia, R

## 4.4 First comparison, before any checking

Put the two side by side and answer one question: **which one could you act on?**

Not which one is more convincing — the memory answer usually reads *better*, because it is not
constrained by eight specific papers. Which one could you check?

In [26]:
# ============================================================================
# 4.3 SIDE BY SIDE
# ============================================================================


def parse_claims(text):
    """Pull (claim, pmid, quote) out of either condition's output.

    Lines that merely echo the format template are skipped. Small models do
    this often: they copy the instruction line, including the literal
    'PMID:12345678', and a naive parser counts it as a citation. That is a
    parser bug, not a model finding, so it is filtered here.

    Three shapes are accepted, because a 4B model will not always obey the
    format it was given:

        CLAIM: ... | PMID:12345678 | QUOTE: ...
        {"claim": "...", "pmid": "...", "quote": "..."}
        1. some claim text ... PMID: 12345678

    The last one is why this function does not require the CLAIM: prefix -
    a numbered list with a PMID in it is still a citable claim.
    """
    rows = []

    # --- shape 1: JSON, whole string or one object per line --------------
    _stripped = text.strip()
    if _stripped.startswith("{") or _stripped.startswith("["):
        try:
            _blob = json.loads(_stripped)
            if isinstance(_blob, dict):
                _blob = _blob.get("claims", [_blob])
            for _o in _blob:
                if not isinstance(_o, dict):
                    continue
                _pm = str(_o.get("pmid") or _o.get("PMID") or "")
                if not re.fullmatch(r"\d{5,9}", _pm):
                    continue
                _cl = str(_o.get("claim") or "").strip()
                if not _cl or "<" in _cl:
                    continue
                rows.append({"claim": _cl[:160], "pmid": _pm,
                             "quote": str(_o.get("quote") or "").strip().strip('"')})
            if rows:
                return rows
        except Exception:
            pass  # fall through to the line parser

    # --- shape 2: one claim per line ------------------------------------
    for line in text.splitlines():
        if "PMID" not in line:
            continue
        if "<" in line and ">" in line:
            continue
        if re.search(r"PMID:12345678\b", line):
            continue
        pm = re.search(r"PMID[:\s]*(\d{5,9})", line)
        if not pm:
            continue
        qm = re.search(r"QUOTE:\s*(.+)$", line, re.I)
        claim = line
        if "CLAIM:" in line:
            claim = line.split("CLAIM:", 1)[1]
        claim = claim.split("|")[0].strip()
        # strip a leading list marker or bullet
        claim = re.sub(r"^[\-\*\u2022]\s*", "", claim)
        claim = re.sub(r"^\d+[\.\)]\s*", "", claim)
        claim = claim.replace("**", "").strip()
        if not claim:
            continue
        rows.append({
            "claim": claim[:160],
            "pmid": pm.group(1),
            "quote": (qm.group(1).strip().strip('"') if qm else ""),
        })
    return rows


A_CLAIMS = parse_claims(A_TEXT)
B_CLAIMS = parse_claims(B_TEXT)

# --- one automatic retry, so a format failure is not a dead end ----------
# A 4B model occasionally ignores the output format entirely. Without this,
# an empty B_CLAIMS makes Parts 5 to 9 unrunnable and the exercise stops.
# The retry asks for the simplest possible shape and is logged like any
# other call, so it stays visible in your prompt log.
B_RETRIED = False
if not B_CLAIMS:
    print("=" * 74)
    print("FORMAT FAILURE - condition B returned nothing parseable.")
    print("Retrying once with a stricter, simpler instruction.")
    print("=" * 74)
    _SIMPLE = (
        "List factual claims from these abstracts. One claim per line.\n"
        "Every line must be exactly:\n"
        "CLAIM: <one sentence> | PMID:<digits> | QUOTE: <words copied from the abstract>\n"
        "No numbering, no bullets, no commentary, no extra text.\n"
        "\nABSTRACTS:\n" + CONTEXT
    )
    B_TEXT, B_SECS, B_INFO = ask(_SIMPLE, num_predict=900)
    B_CLAIMS = parse_claims(B_TEXT)
    B_RETRIED = True
    print()
    print("retry parsed: " + str(len(B_CLAIMS)) + " claim(s)")
    print()

print("=" * 74)
print("%-26s %-14s %-14s" % ("", "A memory", "B grounded"))
print("=" * 74)
print("%-26s %-14d %-14d" % ("claims parsed", len(A_CLAIMS), len(B_CLAIMS)))
print("%-26s %-14d %-14d" % ("citations given", len(A_CLAIMS), len(B_CLAIMS)))
print("%-26s %-14d %-14d" % ("quotes given", 0, sum(1 for c in B_CLAIMS if c["quote"])))
print("%-26s %-14.1f %-14.1f" % ("seconds", A_SECS, B_SECS))
print("=" * 74)
print()
if not B_CLAIMS:
    print("Condition B returned nothing parseable, even after a retry.")
    print()
    print("This is a FORMAT failure, not a knowledge failure. Three things")
    print("to try, in order:")
    print("  1. Use a larger model - set MODEL in cell 0.2 to qwen3.5:9b or")
    print("     qwen2.5:14b-instruct, then re-run from Part 1.")
    print("  2. Raise num_predict in the 4.2 call to 1400 - the answer may")
    print("     have been cut off before any line completed.")
    print("  3. Paste the raw output above into a chat assistant and ask it")
    print("     to reformat as 'CLAIM: ... | PMID: ... | QUOTE: ...' lines,")
    print("     then check each quote yourself. Say that you did this.")
    print()
    print("Parts 5 to 9 need B_CLAIMS. Do not skip them.")
elif B_RETRIED:
    print("The retry worked. Record in your prompt log that condition B")
    print("needed a second, stricter instruction - that is a real finding")
    print("about the model, not a detail to hide.")
print("Ask yourself: of these two, which could you verify? Part 5 does it.")

                           A memory       B grounded    
claims parsed              5              5             
citations given            5              5             
quotes given               0              5             
seconds                    17.4           49.4          

Ask yourself: of these two, which could you verify? Part 5 does it.


> **TASK 2 — memory versus grounded.** Show the class both outputs verbatim (they are short). Then
> walk through three things: (a) which condition produced claims you could check, and why; (b) whether
> the memory answer was more or less convincing to read than the grounded one, and what that tells you
> about judging AI output by fluency; (c) one specific claim from condition A that you suspect is
> wrong, and what you would need in order to test it. Do not look anything up yet — this task is about
> what you can tell *without* checking.

---

# Part 5 — Three levels of verification

## 5.1 Why one check is not enough

It is tempting to check one thing: does the cited PMID exist? That catches almost nothing, because
the model is good at producing identifiers that exist. You need three levels, and each catches
something the previous one missed.

| Tier | Question | Cost | Catches |
|---|---|---|---|
| **1** | Does the PMID exist, and is it in your corpus? | free | invented numbers |
| **2** | Does the quote actually appear in that abstract? | free | paraphrase, misattribution |
| **3** | Does the quoted sentence **assert** the claim? | **your judgement** | everything else |

Tier 3 is the one no code can pass for you. Part 6 shows you why it is unavoidable.

## 5.2 Tier 1 — does the paper exist?

Two different failures get merged if you only check one thing:

- **not in PubMed at all** — a fabricated number;
- **real, but not in your corpus** — the model answered from memory and got a real paper that has
  nothing to do with your topic.

Keep them separate when you present. The first is a hallucination and everyone knows to look
for it; the second is the dangerous one.

In [27]:
# ============================================================================
# 5.1 TIER 1 - DOES THE PMID EXIST?
# ============================================================================

CORPUS_PMIDS = {p["pmid"] for p in USABLE}
BY_PMID = {p["pmid"]: p for p in USABLE}


def tier1(pmid, check_pubmed=True):
    """Return (verdict, title). Checks the corpus first, then PubMed."""
    if not pmid:
        return "no pmid given", ""
    if pmid in CORPUS_PMIDS:
        return "in corpus", BY_PMID[pmid]["title"]
    if not check_pubmed:
        return "not in corpus", ""
    try:
        r = eutils("esummary.fcgi", db="pubmed", id=pmid, retmode="json")
        rec = r.json().get("result", {}).get(pmid)
        if rec and "error" not in rec:
            return "exists, outside corpus", (rec.get("title") or "")[:120]
        return "DOES NOT EXIST", ""
    except Exception:
        return "lookup failed", ""
    finally:
        time.sleep(PAUSE)


print("=" * 78)
print("TIER 1 - DO THE CITED PAPERS EXIST?")
print("=" * 78)

for _label, _claims in (("A memory", A_CLAIMS), ("B grounded", B_CLAIMS)):
    print()
    print("--- " + _label)
    if not _claims:
        print("    (no claims parsed)")
        continue
    _counts = {}
    for _c in _claims:
        _verdict, _title = tier1(_c["pmid"])
        _counts[_verdict] = _counts.get(_verdict, 0) + 1
        print("    %-10s %-24s %s"
              % (_c["pmid"] or "-", _verdict, _title[:56]))
    print()
    for _v, _n in sorted(_counts.items(), key=lambda kv: -kv[1]):
        print("    %-24s %d" % (_v, _n))

print()
print("=" * 78)
print("'exists, outside corpus' is the dangerous row: a real paper, cited")
print("confidently, that your corpus never contained. Look at those titles.")
print("=" * 78)

TIER 1 - DO THE CITED PAPERS EXIST?

--- A memory
    20933314   exists, outside corpus   How children and young people construct and negotiate li
    15768445   exists, outside corpus   Adsorption kinetics and thermodynamics of acid dyes on a
    23942834   exists, outside corpus   Recent developments in manufacturing oligosaccharides wi
    35621089   exists, outside corpus   Outcomes of the two generations of bioresorbable scaffol
    34567890   exists, outside corpus   Multiple Primary Malignancies in Head and Neck Cancer: A

    exists, outside corpus   5

--- B grounded
    36540606   in corpus                Multiple Myeloma with Histiocytosis.
    34659990   in corpus                Cutaneous Multiple Myeloma.
    34659990   in corpus                Cutaneous Multiple Myeloma.
    35454812   in corpus                Metabolic Vulnerabilities in Multiple Myeloma.
    35454812   in corpus                Metabolic Vulnerabilities in Multiple Myeloma.

    in corpus                

## 5.3 Tier 2 — does the quote appear?

For condition B, the model gave you a quote for each claim. Compare it to the abstract, character for
character, after normalising case and punctuation.

Two things make this non-trivial, and both are worth knowing:

- **the quote may be split** across a sentence boundary or a hyphenated line;
- **the model may paraphrase while presenting it as a quote**, which is the interesting failure.

The check below normalises both sides and looks for the quote inside the abstract. It reports the
longest matching prefix so you can see *how far* a near-miss got.

In [28]:
# ============================================================================
# 5.2 TIER 2 - IS THE QUOTE VERBATIM?
# ============================================================================


def tier2(pmid, quote):
    """Return (verdict, detail) for a quote against its cited abstract."""
    if not quote:
        return "no quote given", ""
    if pmid not in BY_PMID:
        return "cannot verify", "pmid not in corpus"
    haystack = normalise(BY_PMID[pmid]["title"] + " " + BY_PMID[pmid]["abstract"])
    needle = normalise(quote)
    if not needle:
        return "no quote given", ""
    if needle in haystack:
        return "verbatim", "found exactly"

    # longest matching run of words, so a partial match is visible
    words = needle.split()
    for size in range(min(len(words), 18), 3, -1):
        for start in range(0, len(words) - size + 1):
            probe = " ".join(words[start:start + size])
            if probe in haystack:
                return ("partial", "%d of %d words match" % (size, len(words)))
    return "NOT FOUND", "no run of 4+ words appears in the abstract"


print("=" * 78)
print("TIER 2 - ARE THE QUOTES VERBATIM?")
print("=" * 78)
print()

if not B_CLAIMS:
    print("  No grounded claims to check. Re-run Part 4.2.")
else:
    _ok = 0
    for _c in B_CLAIMS:
        _verdict, _detail = tier2(_c["pmid"], _c["quote"])
        if _verdict == "verbatim":
            _ok += 1
        print("  %-10s %-12s %s" % (_c["pmid"] or "-", _verdict, _detail))
        print("             claim: " + _c["claim"][:72])
        print("             quote: " + _c["quote"][:72])
        print()
    print("=" * 78)
    print("  verbatim: %d of %d" % (_ok, len(B_CLAIMS)))
    print("=" * 78)
    print()

    # A quote that stops mid-sentence is a symptom of a truncated answer,
    # not of dishonesty. Flag it so it is not misread as a model failure.
    _trunc = [c for c in B_CLAIMS
              if c["quote"] and not re.search(r"[.!?\"')\]]\s*$", c["quote"])]
    if _trunc:
        print("  " + str(len(_trunc)) + " quote(s) do not end at a sentence")
        print("  boundary. If the answer was cut off (done_reason='length' in")
        print("  4.2), that is why - raise num_predict and re-run before you")
        print("  read anything into the tier-2 numbers below.")
        print()

    print("A high verbatim rate means the model is quoting rather than")
    print("paraphrasing. It does NOT mean the claims are supported - that is")
    print("tier 3, and Part 6 shows why you cannot skip it.")

TIER 2 - ARE THE QUOTES VERBATIM?

  36540606   verbatim     found exactly
             claim: Multiple myeloma is a disease characterized by abnormal clonal plasma ce
             quote: Myeloma is a disease characterized by abnormal clonal plasma cells that 

  34659990   verbatim     found exactly
             claim: Multiple myeloma is a plasma cell dyscrasia characterized by abnormal bo
             quote: Multiple myeloma is a plasma cell dyscrasia characterized by abnormal bo

  34659990   verbatim     found exactly
             claim: Organ involvement in multiple myeloma manifests as CRAB (hyperCalcemia, 
             quote: Organ involvement in multiple myeloma manifests as CRAB (hyperCalcemia, 

  35454812   verbatim     found exactly
             claim: Multiple myeloma is an incurable malignancy with eventual emergence of r
             quote: Multiple myeloma (MM) remains an incurable malignancy with eventual emer

  35454812   verbatim     found exactly
             clai

## 5.4 The result you should expect

On the reference run, condition B scored **10 of 10 verbatim**. Every quote was real. Every PMID was
real. And at least one claim was still wrong — which is Part 6's job.

That is the sequence to internalise:

> **A claim can pass tier 1, pass tier 2, and still be false.**

If your run shows the same thing, you have reproduced the central finding of this exercise. If it
does not, say so — a model that paraphrases rather than quotes fails at tier 2 instead, and that is a
different but equally reportable result.

## 5.5 What tier 2 cannot see

Tier 2 checks whether a string appears. It has no idea whether the string is *relevant*. A paper can
contain the exact phrase "angiosarcoma" once, in a list of things the tumour **resembles**, and pass
every mechanical check while supporting the opposite of the claim.

That is Part 6.

> **TASK 3 — tiers 1 and 2.** Take the class through the tier-1 table for both conditions: how many
> PMIDs were in your corpus, how many existed outside it, and how many did not exist at all. Then give
> tier 2's verbatim rate. Finish on the sentence that matters: **how many claims passed both tiers?**
> Do not judge whether they are *true* yet — that is Task 4. Say only what the mechanical checks
> establish, and be explicit about what they do not.

---

# Part 6 — Tier 3, and why you cannot automate it

## 6.1 The case that breaks every mechanical check

The reference run produced this claim for the topic *angiosarcoma*:

| | |
|---|---|
| **Claim** | something about angiosarcoma |
| **Cited PMID** | 42998681 — exists, in corpus |
| **Quote** | verbatim, appears in the abstract |
| **Tier 1** | ✅ pass |
| **Tier 2** | ✅ pass |
| **Tier 3** | ❌ **the paper is not about angiosarcoma** |

The paper is *"Unusual Morphologic Variants of Solitary Fibrous Tumor"*. The word appears once:

> "Histologic patterns included **angiosarcoma-like** (n=2)…"

It is a **histological mimic**. The paper says these tumours *resemble* angiosarcoma, which is close
to the opposite of being about it. A keyword check passes. A named-entity model would tag
`angiosarcoma-like` as a disease and be wrong. Only reading the sentence catches it.

## 6.2 The vocabulary of "not really"

Biomedical text is full of hedges, and each one inverts or weakens a claim. These are the patterns to
look for:

| Pattern | Example | What it does to the claim |
|---|---|---|
| `X-like` | `angiosarcoma-like` | resemblance, not identity |
| `mimicking` / `masquerading as` | `angiosarcoma mimicking carcinoma` | the named disease is *not* the diagnosis |
| `-oid` | `sarcomatoid` | "resembling", not "being" |
| negation | `no evidence of`, `absence of`, `without` | removes the claim entirely |
| differential | `should be distinguished from` | names a disease in order to exclude it |
| hypothetical | `may represent`, `could be consistent with` | possibility, not finding |

This is a named, hard problem in biomedical NLP, and it is why "the LLM found the entity" is never
the end of the story.

## 6.3 The scan

The cell below flags every claim and quote containing one of these patterns, next to the claim it was
used to support. It is a **hint generator, not a judge** — it tells you where to look, and you decide.

In [29]:
# ============================================================================
# 6.3 HEDGE AND NEGATION SCAN
# ============================================================================

HEDGE_PATTERNS = [
    (r"\b\w+-like\b", "X-like: resemblance, not identity"),
    (r"\bmimick?(ing|s)?\b", "mimicking: the named disease is not the diagnosis"),
    (r"\bmasquerad(e|ing|es)\b", "masquerading as: same problem"),
    (r"\b\w+oid\b", "-oid: 'resembling', not 'being'"),
    (r"\bno evidence of\b", "negation"),
    (r"\babsence of\b", "negation"),
    (r"\bwithout\b", "negation"),
    (r"\bnot\b", "negation"),
    (r"\bcannot\b|\bcan not\b", "negation"),
    (r"\bdistinguish(ed)? from\b", "differential: named in order to exclude"),
    (r"\bunlikely\b", "hedge"),
    (r"\bmay (be|represent|suggest)\b", "hedge: possibility"),
    (r"\bcould (be|represent)\b", "hedge: possibility"),
    (r"\bruled out\b", "negation"),
]


def scan_hedges(text):
    """Return [(pattern, meaning, matched_text)]."""
    found = []
    for pattern, meaning in HEDGE_PATTERNS:
        for m in re.finditer(pattern, text, re.I):
            found.append((pattern, meaning, m.group(0)))
    return found


print("=" * 78)
print("HEDGE SCAN - every claim or quote containing a hedge or negation")
print("=" * 78)
print()

_flagged = 0
for _c in B_CLAIMS:
    _hits_q = scan_hedges(_c["quote"] or "")
    _hits_c = scan_hedges(_c["claim"] or "")
    if not _hits_q and not _hits_c:
        continue
    _flagged += 1
    print("PMID " + (_c["pmid"] or "-"))
    print("  claim: " + _c["claim"][:76])
    print("  quote: " + _c["quote"][:76])
    for _p, _meaning, _m in _hits_q:
        print("     FLAG in QUOTE  '" + _m + "'   " + _meaning)
    for _p, _meaning, _m in _hits_c:
        print("     FLAG in CLAIM  '" + _m + "'   " + _meaning)
    print()

print("=" * 78)
print("  claims flagged: %d of %d" % (_flagged, len(B_CLAIMS)))
print("=" * 78)
print()
if _flagged:
    print("Each flagged row needs a human read. The flag says 'look here',")
    print("not 'this is wrong' - 'X-like' in a sentence about a DIFFERENT")
    print("disease is exactly the trap, but the flag cannot tell you which")
    print("disease the sentence was about.")
else:
    print("Nothing flagged. That is unusual for a real abstract corpus -")
    print("check that your quotes are longer than a single clause, because")
    print("a hedge in the sentence BEFORE the quote is invisible here.")
    print("Scan the abstracts yourself in 6.5 and say what you find.")

HEDGE SCAN - every claim or quote containing a hedge or negation

  claims flagged: 0 of 5

Nothing flagged. That is unusual for a real abstract corpus -
check that your quotes are longer than a single clause, because
a hedge in the sentence BEFORE the quote is invisible here.
Scan the abstracts yourself in 6.5 and say what you find.


## 6.4 The other tell: the claim says more than the quote

Hedges are one way a claim outruns its evidence. The other is **added specificity** — the claim names
things the quoted sentence never mentions.

Tier 2 cannot see this at all. It checks that the *quote* appears in the paper. It never checks the
*claim* against the quote, so a model can quote a real sentence honestly and then attach a conclusion
to it that the sentence does not contain. The quote is genuine, the citation is genuine, and the claim
is the model's own training data wearing the paper's clothes.

The cell below does that comparison mechanically: every token in the claim that does not appear in the
quote is flagged. It is crude — it will miss paraphrase, and it will over-flag a claim that legitimately
generalises. It is still the check that catches detail smuggled into a claim its own citation does not
contain.

In [30]:
# ============================================================================
# 6.4 ADDED SPECIFICITY - does the claim contain what the quote does not?
# ============================================================================

_TOKEN_RE = re.compile(
    r"[A-Za-z0-9]+(?:[\-/][A-Za-z0-9]+)*%?")

# Words too common to be evidence of anything.
_GENERIC = {
    "the", "and", "for", "with", "that", "this", "from", "are", "was", "were",
    "has", "have", "had", "its", "their", "which", "who", "not", "but", "can",
    "may", "most", "more", "less", "than", "such", "also", "been", "being",
    "these", "those", "into", "over", "about", "between", "after", "before",
    "using", "used", "use", "only", "both", "each", "other", "some", "any",
    "all", "one", "two", "three", "however", "therefore", "although", "while",
    "associated", "including", "included", "involves", "involving", "cases",
    "patients", "study", "studies", "results", "reported", "showed", "shown",
}

# British/American pairs that are the same word. Without this the checker
# flags 'tumor' against 'tumour' and looks broken.
_SPELLING = {"tumour": "tumor", "tumours": "tumors", "haem": "hem",
             "anaemia": "anemia", "oedema": "edema", "aetiology": "etiology",
             "colour": "color", "behaviour": "behavior", "sarcoma": "sarcoma"}


def _stem(token):
    """A crude stem, enough to stop 'confirmed' matching against 'confirm'.

    This is deliberately shallow. It is not linguistics, it is noise
    reduction - and the docstring in 6.4 says the check is a smoke
    detector, not a proof.
    """
    tok = token.lower().strip("-")
    tok = _SPELLING.get(tok, tok)
    for suffix in ("ations", "ation", "ings", "ing", "edly", "ed", "ly", "es",
                   "s"):
        if len(tok) > len(suffix) + 3 and tok.endswith(suffix):
            tok = tok[:-len(suffix)]
            break
    return tok


def added_specificity(claim, quote):
    """Tokens in the claim that do not appear in the quote.

    The topic itself is excluded: a claim about angiosarcoma that cites a
    paper about angiosarcoma is not 'adding detail' by naming it. Spelling
    variants and simple inflections are normalised away. Tokens shorter
    than four characters are ignored unless they contain a digit - 'of' is
    not evidence, '30%' is.
    """
    if not quote:
        return []
    q_stems = {_stem(t) for t in _TOKEN_RE.findall(quote)}
    topic_stems = {_stem(t) for t in _TOKEN_RE.findall(MY_TOPIC)}
    missing = []
    for tok in _TOKEN_RE.findall(claim):
        low = tok.lower()
        if low in _GENERIC:
            continue
        if len(tok) < 4 and not any(ch.isdigit() for ch in tok):
            continue
        stem = _stem(tok)
        if stem in q_stems or stem in topic_stems:
            continue
        missing.append(tok)
    seen, out = set(), []
    for tok in missing:
        if _stem(tok) not in seen:
            seen.add(_stem(tok))
            out.append(tok)
    return out


print("=" * 78)
print("ADDED SPECIFICITY - the claim contains things the quote does not")
print("=" * 78)
print()

_over = 0
for _c in B_CLAIMS:
    _miss = added_specificity(_c["claim"], _c["quote"])
    if not _miss:
        continue
    _over += 1
    print("PMID " + (_c["pmid"] or "-"))
    print("  claim: " + _c["claim"][:74])
    print("  quote: " + (_c["quote"] or "(none)")[:74])
    print("  NOT IN THE QUOTE: " + ", ".join(_miss[:10]))
    print()

print("=" * 78)
print("  claims carrying extra detail: %d of %d" % (_over, len(B_CLAIMS)))
print("=" * 78)
print()
if _over:
    print("Each row is a claim asserting something its own citation does not")
    print("say. Some will be legitimate generalisation; some will be the model")
    print("adding what it knows from training. You cannot tell which from this")
    print("output - that is what 6.5 is for.")
else:
    print("No claim outran its quote. That is a good sign and worth reporting,")
    print("but it does not mean the claims are true - a claim can be entirely")
    print("contained in a quote from a paper about something else.")

ADDED SPECIFICITY - the claim contains things the quote does not

PMID 36540606
  claim: Multiple myeloma is a disease characterized by abnormal clonal plasma cell
  quote: Myeloma is a disease characterized by abnormal clonal plasma cells that pr
  NOT IN THE QUOTE: produce

PMID 35454812
  claim: Metabolic shifts, which ensure the availability of sufficient energy to su
  quote: Metabolic shifts, which ensure the availability of sufficient energy to su
  NOT IN THE QUOTE: myel

  claims carrying extra detail: 2 of 5

Each row is a claim asserting something its own citation does not
say. Some will be legitimate generalisation; some will be the model
adding what it knows from training. You cannot tell which from this
output - that is what 6.5 is for.


## 6.5 Do the tier-3 read

You now have everything needed to do the check no code can do. For each claim:

1. Open the abstract (the cell below prints it).
2. Read the quoted sentence **in its paragraph**.
3. Decide: does this sentence assert the claim, or merely mention the topic?

Score each claim `supported` / `partly` / `not supported` / `off-topic paper`. Write your criterion
down **before** you start, or you will drift toward accepting whatever the model wrote — which is the
exact failure this exercise is about.

In [31]:
# ============================================================================
# 6.5 THE TIER-3 WORKSHEET - read each quote in context
# ============================================================================

for _c in B_CLAIMS:
    _pmid = _c["pmid"]
    print("=" * 78)
    print("CLAIM: " + _c["claim"][:76])
    print("PMIDs: " + (_pmid or "-"))
    if _pmid in BY_PMID:
        _p = BY_PMID[_pmid]
        print("TITLE: " + _p["title"][:76])
        print("-" * 78)
        print("QUOTE: " + (_c["quote"] or "(none)")[:200])
        print("-" * 78)
        # show the sentences around the quote, so context is visible
        _sentences = re.split(r"(?<=[.!?])\s+", _p["abstract"])
        _needle = normalise((_c["quote"] or "")[:60])
        _shown = 0
        for _s in _sentences:
            if _needle and _needle[:40] in normalise(_s):
                for _near in _sentences[max(0, _sentences.index(_s) - 1):
                                        _sentences.index(_s) + 2]:
                    print("   ... " + _near.strip()[:180])
                    _shown += 1
                break
        if not _shown:
            print("   (quote not located sentence-by-sentence - read the abstract)")
    else:
        print("  PMID not in corpus - cannot read it here.")
    print()
    print("  VERDICT (write it down): supported / partly / not supported / off-topic")
    print()

print("=" * 78)
print("Take this screen with you - it is everything you need to present.")
print("=" * 78)

CLAIM: Multiple myeloma is a disease characterized by abnormal clonal plasma cells 
PMIDs: 36540606
TITLE: Multiple Myeloma with Histiocytosis.
------------------------------------------------------------------------------
QUOTE: Myeloma is a disease characterized by abnormal clonal plasma cells that produces paraprotein.
------------------------------------------------------------------------------
   ... Myeloma is a disease characterized by abnormal clonal plasma cells that produces paraprotein.
   ... The coincidence of myeloma and histiocytosis is a rare clinical manifestation.

  VERDICT (write it down): supported / partly / not supported / off-topic

CLAIM: Multiple myeloma is a plasma cell dyscrasia characterized by abnormal bone m
PMIDs: 34659990
TITLE: Cutaneous Multiple Myeloma.
------------------------------------------------------------------------------
QUOTE: Multiple myeloma is a plasma cell dyscrasia characterized by abnormal bone marrow clonal plasma cells, histologic

> **TASK 4 — tier 3, the read.** Walk the class through your table: each claim, its PMID, the tier-1
> and tier-2 verdicts, your **added-specificity** flags from 6.4, and your tier-3 verdict with a
> one-line justification. Then the number that matters: **how many claims passed tiers 1 and 2 but
> failed tier 3?** For each one, read out the specific words that changed your mind — `-like`,
> `mimicking`, a negation, an off-topic paper, or a detail the quote does not contain. If every claim
> genuinely held up, say so and show the evidence, because that is a strong claim and your audience
> should push back on it.

---

# Part 7 — Your Exercise 3 linker becomes the referee

## 7.1 The idea

So far you have checked whether the *paper* supports the *claim*. Now check whether the *entities* the
model extracted are real.

This is where Exercise 3 pays off. The model will name diseases, genes, and phenotypes in its claims.
You push each name through `resolve_mention`, and the cascade you built decides:

- `linked_label` — exact match, real identifier;
- `linked_synonym` — real, matched through a curator's synonym;
- `ambiguous` — several terms share the name, so **no link**;
- `fuzzy` — close, but the cascade **refuses** to link it;
- `unlinked` — not in the vocabulary at all.

That last group is the interesting one. A mention the model used confidently that your vocabulary does
not contain is either a different granularity, a different vocabulary, or an invention.

## 7.2 Extracting the mentions

Rather than a full NER model, the cell below does something more transparent: it looks for **your
ontology's own labels and synonyms** inside the claims, plus the topic itself. That is a fair
approximation of what a named-entity model would surface, and it makes every step inspectable.

## 7.3 Why this is not circular

It is a fair objection: if you search for the vocabulary's own labels, of course they resolve. Two
answers.

First, the model's phrasing will not match the vocabulary exactly — it says "cancer of the blood
vessels", not `angiosarcoma` — and those misses are the finding.

Second, and more importantly, the linker's job here is to catch the **confident near-miss**. When the
model writes `angiosarcoma-like`, the linker should refuse it, and that refusal is information.

In [32]:
# ============================================================================
# 7.1 RESOLVE EVERY MENTION THE MODEL MADE
# ============================================================================

# Candidate mentions come from three places:
#   1. your vocabulary's labels that appear in the claims;
#   2. your vocabulary's synonyms that appear in the claims;
#   3. the topic itself, plus the short phrases the model built around it.
#
# (3) is the interesting one. It is where 'angiosarcoma-like',
# 'angiosarcoma of the spleen' and 'angiosarcoma is a rare' land, and the
# linker's verdict on each of them is the result.

ALL_CLAIMS = A_CLAIMS + B_CLAIMS
CLAIM_TEXT = " ".join(c["claim"] for c in ALL_CLAIMS)
CLAIM_LOWER = normalise(CLAIM_TEXT)

# Words that can never end a noun phrase. Used to trim the model's phrasing
# back to something a vocabulary could plausibly contain.
_STOP_TAIL = {
    "is", "are", "was", "were", "be", "been", "being", "the", "a", "an",
    "of", "in", "on", "at", "to", "for", "with", "and", "or", "but", "as",
    "by", "from", "that", "which", "who", "its", "their", "has", "have",
    "had", "can", "may", "might", "often", "also", "rarely", "usually",
    "typically", "commonly", "most", "more", "less", "very", "rare",
    "rarely", "accounts", "arises", "occur", "occurs", "affects",
}


def _trim_phrase(words):
    """Cut the phrase at the first function word, then drop trailing glue.

    'angiosarcoma is a malignant' -> 'angiosarcoma'
    'angiosarcoma-like'           -> 'angiosarcoma-like'  (one token, survives)
    'angiosarcoma of the spleen'  -> 'angiosarcoma of spleen'

    A preposition is allowed to survive mid-phrase so that the more specific
    forms - the ones that test 'different granularity' in 7.4 - are kept.
    """
    out = []
    for i, word in enumerate(words):
        bare = word.lower().strip(".,;:()")
        if bare in _STOP_TAIL:
            if bare in ("of", "in") and i > 0 and len(out) < 3:
                out.append(word)
                continue
            break
        out.append(word)
    while out and out[-1].lower().strip(".,;:()") in _STOP_TAIL:
        out.pop()
    return out


# Word-boundary matching. A bare `in` test is wrong here: DOID carries the
# synonym 'rMED', and 'rmed' is a substring of 'confirmed', so a naive
# substring search links the word 'confirmed' to spinal muscular atrophy.
def _appears(needle, haystack):
    return re.search(r"(?<![a-z0-9])" + re.escape(needle) + r"(?![a-z0-9])",
                     haystack) is not None


MENTIONS = {}
for _label in LABEL_TO_IDS:
    if len(_label) > 3 and _appears(_label, CLAIM_LOWER):
        MENTIONS[normalise(_label)] = _label
for _syn in SYNONYM_TO_IDS:
    if len(_syn) > 3 and _appears(_syn, CLAIM_LOWER):
        MENTIONS.setdefault(normalise(_syn), _syn)

# Phrases built around the topic. Work on the RAW claim text so hyphenated
# forms ('angiosarcoma-like') survive; resolve_mention normalises internally.
_TOPIC_RE = re.compile(re.escape(MY_TOPIC) + r"(?:[-\s][A-Za-z][\w-]*){0,4}",
                       re.I)
for _m in _TOPIC_RE.finditer(CLAIM_TEXT):
    _phrase = " ".join(_trim_phrase(_m.group(0).split())).strip(" .,;:")
    if 4 < len(_phrase) < 60:
        MENTIONS.setdefault(normalise(_phrase), _phrase)

MENTIONS = sorted(MENTIONS.values())
print("mentions found in the model's claims: " + str(len(MENTIONS)))
print()

print("=" * 78)
print("%-42s %-18s %s" % ("MENTION", "STATUS", "EVIDENCE"))
print("=" * 78)
_resolved = _refused = 0
for _m in MENTIONS:
    _status, _tid, _why = resolve_mention(_m)
    if _status.startswith("linked"):
        _resolved += 1
    else:
        _refused += 1
    print("%-42s %-18s %s" % (_m[:42], _status, _why[:26]))
print("=" * 78)
print()
print("  resolved to an identifier: " + str(_resolved))
print("  refused or unmatched:      " + str(_refused))
print()
print("The refused rows are the result. For each one, say in Task 5 whether")
print("the vocabulary genuinely lacks the concept, or whether the model used")
print("a different granularity than the ontology does.")

mentions found in the model's claims: 11

MENTION                                    STATUS             EVIDENCE
anemia                                     linked_label       exact label
cancer                                     linked_label       exact label
disease                                    linked_label       exact label
hematologic malignancy                     linked_synonym     exact synonym
hypercalcemia                              linked_label       exact label
multiple myeloma                           linked_label       exact label
multiple myeloma manifests                 unlinked           no candidate
myeloma                                    linked_synonym     exact synonym
plasma cell dyscrasia                      linked_synonym     exact synonym
plasmacytoma                               linked_label       exact label
renal insufficiency                        linked_synonym     exact synonym

  resolved to an identifier: 10
  refused or unmatched:      1


## 7.4 What the refusals mean

Sort the refused mentions into three buckets — this is the analysis, not the count. The examples below
are what the DOID linker actually does with each form, measured:

| Bucket | Example | Linker says | What it means |
|---|---|---|---|
| **Hedged form** | `angiosarcoma-like` | `unlinked` | the model echoed the paper's hedge; the linker correctly refused |
| **Inflected form** | `angiosarcomas` | `fuzzy` 0.96 → **refused** | plural of a real term; the cascade will not link a near-miss |
| **Different granularity** | `primary jejunal angiosarcoma` | `unlinked` | a real, more specific concept your vocabulary does not carry |
| **Not in vocabulary** | a gene symbol, a drug name | `unlinked` | wrong vocabulary — your ontology does not cover it |

Note the useful asymmetry: `angiosarcoma of spleen` **does** link, to `DOID:265`, because DOID carries
it as a synonym. So the vocabulary sometimes subdivides and sometimes does not, and you cannot tell
which from the string alone — only by asking the linker.

The first bucket is the one that connects back to Part 6. If your linker refuses `angiosarcoma-like`,
it is doing exactly what tier 3 asked you to do by hand: refusing to treat resemblance as identity.

That is the sentence worth writing down. **Your Exercise 3 code and your Exercise 4 judgement are the
same operation.**

> **TASK 5 — resolution through your own linker.** Show the resolution table, then sort the refused
> mentions into the buckets in 7.4 and give the count for each. Answer the question your audience will
> ask: **did the linker refuse anything it should have linked, or link anything it should have
> refused?** Give one example either way. Finish with the connection — which refused mention is the
> same failure tier 3 caught, and why does that make the linker a usable automated check?

---

# Part 8 — Feed the failure back

## 8.1 Does the model correct itself?

Take the worst claim from your tier-3 table — the one where the quote was real but the paper did not
support it — and tell the model, precisely, what went wrong. Then see what it does.

Three possible behaviours, and they mean different things:

| Behaviour | What it tells you |
|---|---|
| **corrects** — withdraws the claim, offers a better one | the model can be steered with evidence |
| **qualifies** — keeps the claim, adds a hedge | partly fixed; the claim is still there |
| **digs in** — restates the claim, or invents support | the failure is not fixable by prompting |

This is the difference between *using* AI and *trusting* it. A tool that corrects when shown evidence
is usable in a loop. A tool that defends a wrong answer needs a human after every step.

## 8.2 The prompt

Be specific. Do not say "you were wrong" — quote the sentence and say what it actually means. A vague
challenge produces a vague apology, which teaches nothing.

In [33]:
# ============================================================================
# 8.1 THE RETRY LOOP
# ============================================================================

# Choose the claim to challenge. Default: the first grounded claim.
CHALLENGE_INDEX = 0

if not B_CLAIMS:
    print("No grounded claims - run Part 4.2 first.")
else:
    _c = B_CLAIMS[min(CHALLENGE_INDEX, len(B_CLAIMS) - 1)]
    _pmid = _c["pmid"]
    _abstract = BY_PMID.get(_pmid, {}).get("abstract", "")

    print("=" * 78)
    print("CHALLENGED CLAIM")
    print("=" * 78)
    print("  claim: " + _c["claim"][:76])
    print("  pmid:  " + _pmid)
    print("  quote: " + _c["quote"][:76])
    print()

    CHALLENGE = (
        "You made this claim about " + MY_TOPIC + ":\n"
        "\n"
        "CLAIM: " + _c["claim"] + "\n"
        "CITING PMID: " + _pmid + "\n"
        "WITH THE QUOTE: " + _c["quote"] + "\n"
        "\n"
        "Here is the full abstract of PMID " + _pmid + ":\n"
        "\n" + _abstract[:1600] + "\n"
        "\n"
        "Read the abstract carefully. Does it actually support the claim as\n"
        "you stated it? If it does not, say so and withdraw the claim. If it\n"
        "supports a weaker version, state the weaker version. Do not invent\n"
        "new citations.\n"
    )

    RETRY_TEXT, RETRY_SECS, RETRY_INFO = ask(CHALLENGE, num_predict=600)
    print("=" * 78)
    print("THE MODEL'S RESPONSE")
    print("=" * 78)
    print(RETRY_TEXT.strip())
    print()
    print("=" * 78)

    # Crude but useful signal
    _lower = RETRY_TEXT.lower()
    if any(w in _lower for w in ("withdraw", "incorrect", "i was wrong",
                                 "does not support", "doesn't support",
                                 "not supported", "apologi")):
        print("  SIGNAL: appears to concede. Check whether it actually withdrew")
        print("  the claim or merely restated it more carefully.")
    elif any(w in _lower for w in ("supports", "consistent with", "indeed",
                                   "as stated")):
        print("  SIGNAL: appears to defend the claim. Read its reasoning - if it")
        print("  argues the quote does support the claim, that is a finding.")
    else:
        print("  SIGNAL: unclear. Read it yourself; this is tier 3 again.")

CHALLENGED CLAIM
  claim: Multiple myeloma is a disease characterized by abnormal clonal plasma cells 
  pmid:  36540606
  quote: Myeloma is a disease characterized by abnormal clonal plasma cells that prod

THE MODEL'S RESPONSE
The provided abstract **does not** fully support your original claim because of a grammatical inconsistency in the source text itself regarding the verb form ("produces" vs "produce").

In your claim, you used the plural verb "**produce**" to agree with the plural subject "cells." However, the quote from PMID 36540606 uses the singular verb "**produces**":
> "Myeloma is a disease characterized by abnormal clonal plasma cells that **produces** paraprotein."

While the biological fact remains true (the cells produce the protein), the specific text you cited contains a grammatical error where the subject of the relative clause ("cells") does not agree with the verb ("produces"). Because the citation explicitly states "produces" rather than "produce," it technicall

> **TASK 6 — the retry loop.** Show the claim you challenged, the exact challenge prompt, and the
> model's response verbatim. Then classify the behaviour as **corrects / qualifies / digs in**, and
> justify it with a quote. Finish with the practical question, answered plainly: **would you put this
> model in a loop that runs without a human checking each step?** Yes or no, and say what in your own
> results decided it.

---

# Part 9 — The evidence table

## 9.1 What you actually produced

Everything so far has been a check. This part assembles the checks into the one artefact a researcher
would act on: a table where every row can be defended.

| Column | Comes from |
|---|---|
| claim | condition B output |
| pmid | condition B output |
| tier 1 | does the paper exist and is it in the corpus |
| tier 2 | is the quote verbatim |
| tier 3 | does the sentence assert the claim (**you**) |
| resolved id | the Exercise 3 linker |

A row that passes all six is a claim you could put in a paper. A row that fails any of them is a claim
you now know how to reject — and knowing how to reject is the skill.

In [34]:
# ============================================================================
# 9.1 BUILD AND SAVE THE EVIDENCE TABLE
# ============================================================================

# Tier-3 verdicts are yours. Fill them in after the Part 6 read.
# Allowed values: supported / partly / not supported / off-topic paper
#
# Keys are the PMID plus a content fingerprint rather than the raw claim, so
# trivial wording differences (punctuation, case, filler words) do not break
# the match. A genuine rewrite still changes the key - no automatic scheme can
# tell a paraphrase from a different claim, and silently attaching your old
# verdict to a new claim would be worse than asking you again.
#
# What protects the work is the file: the dict is saved to
# tier3_<digit>_<vocab>.json and reloaded here, so a kernel restart or a
# re-run of Part 4 costs you a copy-paste, never your reading.

TIER3 = {
    # "42998681|angiosarcoma-like-patterns-seen-tumours-vascular": "off-topic paper",
    # "40881117|angiosarcoma-jejunal-primary-rare-tumour-vascular": "supported",
}

_STOP = {
    "that", "this", "with", "from", "were", "have", "been", "which", "their",
    "there", "these", "those", "into", "than", "then", "also", "such", "more",
    "most", "some", "when", "what", "will", "would", "could", "should", "about",
    "after", "before", "between", "other", "both", "each", "only", "very",
    "study", "studies", "patient", "patients", "result", "results", "paper",
    "papers", "abstract", "abstracts", "showed", "shows", "found", "using",
}


def claim_key(pmid, claim):
    """Stable key: PMID + content words, order-independent."""
    _w = [w for w in re.findall(r"[a-z0-9]+", (claim or "").lower())
          if len(w) > 3 and w not in _STOP]
    _w = sorted(set(_w))[:6]
    if not _w:
        return str(pmid) + "|" + (claim or "")[:30].strip().lower()
    return str(pmid) + "|" + "-".join(_w)


TIER3_FILE = "tier3_" + str(MY_DIGIT) + "_" + MY_VOCAB + ".json"

# Load anything you already graded, so a re-run cannot lose it.
if os.path.exists(TIER3_FILE):
    try:
        with io.open(TIER3_FILE, encoding="utf-8") as _fh:
            _saved = json.load(_fh)
        TIER3.update(_saved)
        print("loaded " + str(len(_saved)) + " saved verdict(s) from " + TIER3_FILE)
    except Exception as _exc:
        print("could not read " + TIER3_FILE + ": " + str(_exc))
else:
    print("no " + TIER3_FILE + " yet - this is your first run.")


def mentions_in(text):
    """Topic-anchored phrases in one claim, trimmed to content words."""
    found = []
    for _m in _TOPIC_RE.finditer(text):
        _p = " ".join(_trim_phrase(_m.group(0).split())).strip(" .,;:")
        if 4 < len(_p) < 60 and _p not in found:
            found.append(_p)
    return found


def resolve_ids(text):
    """Resolve every topic mention in a claim; return 'MENTION=ID; ...'."""
    parts = []
    for _m in mentions_in(text):
        _status, _tid, _why = resolve_mention(_m)
        if _status.startswith("linked"):
            parts.append(_m + "=" + str(_tid))
        else:
            parts.append(_m + "=" + _status)
    return "; ".join(parts)[:150]


_rows = []
for _c in B_CLAIMS:
    _key = claim_key(_c["pmid"], _c["claim"])
    _t1, _t1_title = tier1(_c["pmid"], check_pubmed=False)
    _t2, _t2_detail = tier2(_c["pmid"], _c["quote"])
    _rows.append({
        "claim": _c["claim"][:110],
        "pmid": _c["pmid"],
        "tier1": _t1,
        "tier2": _t2,
        "tier3": TIER3.get(_key, ""),
        "resolved": resolve_ids(_c["claim"]),
        "quote": _c["quote"][:150],
    })

EVIDENCE = pd.DataFrame(_rows)
EVIDENCE_FILE = "evidence_" + str(MY_DIGIT) + "_" + MY_VOCAB + ".csv"
EVIDENCE.to_csv(EVIDENCE_FILE, index=False)

# Save the verdicts themselves, so re-running Part 4 or restarting the
# kernel cannot throw away your reading work.
try:
    with io.open(TIER3_FILE, "w", encoding="utf-8") as _fh:
        json.dump(TIER3, _fh, indent=1, ensure_ascii=False, sort_keys=True)
except Exception as _exc:
    print("could not save " + TIER3_FILE + ": " + str(_exc))

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 46)
print(EVIDENCE.to_string(index=False))
print()
print("saved: " + EVIDENCE_FILE + "   " + TIER3_FILE)
print()
_unfilled = int((EVIDENCE["tier3"] == "").sum())
if _unfilled:
    print(str(_unfilled) + " row(s) still have no tier-3 verdict.")
    print()
    print("Add them to the TIER3 dict above, using the key shown here, then")
    print("re-run this cell. Your keys are:")
    for _c in B_CLAIMS:
        if not TIER3.get(claim_key(_c["pmid"], _c["claim"]), ""):
            print('    "' + claim_key(_c["pmid"], _c["claim"]) + '": "",')
    print()

    # If a previous run left verdicts for the same PMIDs, show them next to
    # the new claims. A reworded claim cannot be matched automatically - that
    # would risk attaching the wrong verdict - but re-applying a verdict you
    # already made should take seconds, not another read of ten abstracts.
    _orphans = {}
    for _k, _v in TIER3.items():
        _p = _k.split("|", 1)[0]
        if _v and not any(TIER3.get(claim_key(c["pmid"], c["claim"]), "")
                          for c in B_CLAIMS if c["pmid"] == _p):
            _orphans.setdefault(_p, []).append((_k, _v))
    if _orphans:
        print("You have verdicts from an earlier run for these PMIDs, but the")
        print("claim wording changed, so they no longer match:")
        print()
        for _p, _items in sorted(_orphans.items()):
            for _k, _v in _items:
                print("    %-9s %-16s %s" % (_p, _v, _k.split("|", 1)[1]))
        print()
        print("Re-apply them to the new keys above. This is why the dict is")
        print("saved to " + TIER3_FILE + " - a re-run of Part 4 costs you a")
        print("copy-paste, not your reading work.")
    else:
        print("If you re-ran Part 4 and your claims changed, your old verdicts")
        print("are still in " + TIER3_FILE + " - open it and copy them across.")
else:
    print("All rows have a tier-3 verdict. That is your evidence table.")

no tier3_2_doid.json yet - this is your first run.
                                                                                                         claim     pmid     tier1    tier2 tier3                            resolved                                                                                                                                                  quote
         Multiple myeloma is a disease characterized by abnormal clonal plasma cells that produce paraprotein. 36540606 in corpus verbatim                Multiple myeloma=DOID:9538                                                          Myeloma is a disease characterized by abnormal clonal plasma cells that produces paraprotein.
Multiple myeloma is a plasma cell dyscrasia characterized by abnormal bone marrow clonal plasma cells, histolo 34659990 in corpus verbatim                Multiple myeloma=DOID:9538 Multiple myeloma is a plasma cell dyscrasia characterized by abnormal bone marrow clonal plasma cells, h

## 9.2 The summary you will present

The final cell prints one screen with every number you need when you stand up in class. Run it last,
after you have filled in the tier-3 verdicts.

In [35]:
# ============================================================================
# 9.2 FINAL SUMMARY
# ============================================================================

CORPUS_FILE = "abstracts_" + str(MY_DIGIT) + "_" + MY_VOCAB + ".csv"
pd.DataFrame(USABLE).to_csv(CORPUS_FILE, index=False)

print("=" * 70)
print("EXERCISE 4 SUMMARY")
print("=" * 70)
print("  digit / topic / vocab:  " + str(MY_DIGIT) + "  " + MY_TOPIC
      + "  " + MY_VOCAB.upper())
print("  model:                  " + MODEL)
print()
print("  corpus:                 " + str(len(USABLE)) + " abstracts of "
      + str(TOTAL_HITS) + " hits")
print("  condition A claims:     " + str(len(A_CLAIMS)))
print("  condition B claims:     " + str(len(B_CLAIMS)))
print()
if len(EVIDENCE):
    print("  tier 1 in corpus:       "
          + str(int((EVIDENCE["tier1"] == "in corpus").sum())) + "/" + str(len(EVIDENCE)))
    print("  tier 2 verbatim:        "
          + str(int((EVIDENCE["tier2"] == "verbatim").sum())) + "/" + str(len(EVIDENCE)))
    _t3 = EVIDENCE["tier3"].fillna("")
    print("  tier 3 supported:       "
          + str(int((_t3 == "supported").sum())) + "/" + str(len(EVIDENCE)))
    print("  passed all three:       "
          + str(int(((EVIDENCE["tier1"] == "in corpus")
                     & (EVIDENCE["tier2"] == "verbatim")
                     & (_t3 == "supported")).sum())) + "/" + str(len(EVIDENCE)))
print()
print("  mentions refused by the linker: " + str(_refused))
print()
print("  files: " + CORPUS_FILE + "   " + EVIDENCE_FILE)
print("=" * 70)
print()
print("The number that matters is the last row above: how many claims")
print("survived every check. Everything else is how you got there.")

EXERCISE 4 SUMMARY
  digit / topic / vocab:  2  multiple myeloma  DOID
  model:                  qwen3.5:4b

  corpus:                 35 abstracts of 48028 hits
  condition A claims:     5
  condition B claims:     5

  tier 1 in corpus:       5/5
  tier 2 verbatim:        5/5
  tier 3 supported:       0/5
  passed all three:       0/5

  mentions refused by the linker: 1

  files: abstracts_2_doid.csv   evidence_2_doid.csv

The number that matters is the last row above: how many claims
survived every check. Everything else is how you got there.


## 9.3 Your prompt log

`ask()` has been writing every call to `PROMPT_LOG` since Part 1. Nothing extra to do — the cell below
exports it.

This matters more than it sounds. A result you cannot reproduce is an anecdote, and the single thing
that makes an AI-assisted finding reproducible is the **exact prompt**. Not "I asked it to summarise
the papers" — the string, with the temperature and the seed, so someone else can send the same bytes to
the same model tag and get the same answer.

Your log will contain six calls on a clean run: the three determinism runs, condition A, condition B,
and the retry. If condition B failed to parse and the automatic retry fired, there will be **seven** —
and the extra row is a finding, not a mess to tidy up. Report it.

In [36]:
# ============================================================================
# 9.3 EXPORT THE PROMPT LOG
# ============================================================================

LOG_FILE = "prompt_log_" + str(MY_DIGIT) + "_" + MY_VOCAB + ".csv"

# Keep the log readable: truncate the prompt and response columns, but keep
# enough that the call is identifiable.
_log_rows = []
for _e in PROMPT_LOG:
    _log_rows.append({
        "n": _e["n"],
        "model": _e["model"],
        "temp": _e["temperature"],
        "seed": _e["seed"],
        "num_predict": _e["num_predict"],
        "seconds": _e["seconds"],
        "out_tokens": _e["out_tokens"],
        "done_reason": _e["done_reason"],
        "prompt_chars": len(_e["prompt"]),
        "response_chars": len(_e["response"]),
        "prompt_head": _e["prompt"][:300].replace("\n", " "),
        "response_head": _e["response"][:300].replace("\n", " "),
    })

PROMPT_LOG_DF = pd.DataFrame(_log_rows)
PROMPT_LOG_DF.to_csv(LOG_FILE, index=False)

print("=" * 78)
print("PROMPT LOG - " + str(len(PROMPT_LOG)) + " calls")
print("=" * 78)
for _e in PROMPT_LOG:
    print("  call %d  %-12s T=%.1f seed=%s  %5.1fs  %s tok  %s"
          % (_e["n"], _e["model"], _e["temperature"], _e["seed"],
             _e["seconds"], _e["out_tokens"], _e["done_reason"]))
    print("           prompt:   " + _e["prompt"][:66].replace("\n", " ") + " ...")
    print("           response: " + _e["response"][:66].replace("\n", " ") + " ...")
print("=" * 78)
print()
print("saved: " + LOG_FILE)
print()
print("The full prompt and response text is in the CSV? No - only the first")
print("300 characters of each. If you want the complete prompts, change the")
print("[:300] slices above. The point of the log is that a reader can see")
print("WHAT was asked and WHEN, not that it is a full transcript.")

PROMPT LOG - 9 calls
  call 1  qwen3.5:4b   T=0.0 seed=42    3.4s  31 tok  stop
           prompt:   In one sentence, define 'xanthelasma'. No preamble. ...
           response: Xanthelasma is a benign skin condition characterized by the presen ...
  call 2  qwen3.5:4b   T=0.0 seed=42    2.9s  31 tok  stop
           prompt:   In one sentence, define 'xanthelasma'. No preamble. ...
           response: Xanthelasma is a benign skin condition characterized by the presen ...
  call 3  qwen3.5:4b   T=0.0 seed=42    3.0s  31 tok  stop
           prompt:   In one sentence, define 'xanthelasma'. No preamble. ...
           response: Xanthelasma is a benign skin condition characterized by the presen ...
  call 4  qwen3.5:4b   T=0.0 seed=42   17.6s  177 tok  stop
           prompt:   You are a biomedical researcher. Summarise what is known about ric ...
           response: Rickets is a childhood disorder characterized by defective mineral ...
  call 5  qwen3.5:4b   T=0.0 seed=42   58.5s  361 t

> **TASK 7 — the prompt log.** Show your log table (call number, model, temperature, seed, seconds,
> tokens, done_reason — six rows on a clean run, seven if the format retry fired). Then take three
> questions: (a) which single field in that table would change if someone else ran your notebook and
> got different numbers, and why; (b) the determinism test in Part 1.3 sent the same prompt three times
> — why does **that** belong in the log rather than being something you just mention; (c) if you handed
> this experiment to another student, what would they need beyond this log to reproduce your corpus? Be
> specific — it is more than the prompt.

---

## A note on what is on your disk

Four files, all yours:

- `abstracts_<digit>_<vocab>.csv` — the corpus you built
- `evidence_<digit>_<vocab>.csv` — the evidence table, with your tier-3 verdicts
- `prompt_log_<digit>_<vocab>.csv` — every call you made, with its settings
- `cache/<vocab>.obo` — the ontology, re-downloadable and never part of your results

Nothing is uploaded and nothing is collected. The model ran on your machine, against your files.

## The five sentences worth remembering

1. **A model without your documents is answering from training data**, and training data owes nothing
   to your release, your query, or your topic.
2. **A real identifier attached to an irrelevant paper is more dangerous than a fabricated one**,
   because it survives the check you were going to run.
3. **A claim can pass tier 1, pass tier 2, and still be false** — verbatim is not support.
4. **Resemblance is not identity**, and `angiosarcoma-like` is the whole exercise in one hyphen.
5. **A tool you can correct with evidence is usable in a loop; a tool that defends a wrong answer needs
   a human after every step.** Part 8 told you which one you have.

## Where this goes next

Next week runs named-entity recognition over a corpus like yours — GLiNER, spaCy, and a local model —
to find every mention automatically instead of the handful this exercise looked at.

Two things you built today make that possible. You know how to build a corpus that is worth running
NER over, and you know that **finding a mention is the easy half**. The hard half is Part 6: deciding
whether the sentence says what the mention implies.